# detKLDM-Wyckoff-CPR Benchmark


In [ ]:
import dataclasses
import math
import os
import random
import sys
import time
import traceback
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.algorithm10_casal_chart import _decode_lattice_matrix, _encode_lattice_matrix, _k_to_l, _l_to_k
from kldmPlus.algorithm13_fixed_template_velocity_casal import (
    FixedTemplateVelocityConfig,
    compute_template_jacobian,
    finite_difference_jacobian_error,
    materialize_template,
    project_to_fixed_template_local,
    site_row_slices,
    theta_column_slices,
    wrap_residual,
)
from kldmPlus.algorithm16_kldm_wyckoff_ppr_faithful import (
    DIFFCSPPP_CLSMP_GUIDED_MLP_PRESENT,
    DIFFCSPPP_CLSMP_GUIDED_MLP_REASON,
    KLDMGraphState,
    KLDMCleanEstimate,
    PPR_EXACT_CLEAN_V0_REASON,
    PPR_EXACT_CLEAN_V0_SUPPORTED,
    PPR_CLEAN_ESTIMATOR_SOURCE,
    PPR_CLEAN_ESTIMATOR_REASON,
    PPR_LATTICE_POLICY,
    PPR_LATTICE_POLICY_REASON,
    deterministic_predictor_clean_estimate,
    kldm_forward_renoise_fv_only,
    project_clean_to_fixed_wyckoff,
)
from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldm.sample_evaluation import evaluate_csp_reconstruction as evaluate_csp_reconstruction_kldm
from kldmPlus.sample_evaluation import evaluate_csp_reconstruction as evaluate_csp_reconstruction_kldmplus
from kldmPlus.symmetry import select_requested_template_states, validate_requested_space_group
from kldmPlus.symmetry.wyckoff_templates import conventional_cell_multiplicity, expand_wyckoff_template_torch
from kldmPlus.symmetry.pcs_projection import _build_vanilla_structure, _periodic_pairwise_distances, _species_assignment_indices
from kldmPlus.utils.time import iter_sampling_times, sampling_grid

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)
COMPARE_CFG = CONFIG['sampling_compare']
PROFILE = os.environ.get('KLDM_WYCKOFF_PPR_PROFILE', 'laptop').strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop


def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]


SAMPLE_SEED = int(profile_default('KLDM_WYCKOFF_PPR_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_WYCKOFF_PPR_GRAPH_IDS', '2,3,4,5', '2,3,4,5'))
CAPTURE_N_STEPS = int(profile_default('KLDM_WYCKOFF_PPR_CAPTURE_N_STEPS', '1000', '1000'))
CAPTURE_STEP = int(profile_default('KLDM_WYCKOFF_PPR_CAPTURE_STEP', '900', '900'))
CLEAN_ESTIMATOR_STEPS = int(profile_default('KLDM_WYCKOFF_PPR_CLEAN_ESTIMATOR_STEPS', '16', '24'))
CLEAN_ESTIMATOR_T_FINAL = float(profile_default('KLDM_WYCKOFF_PPR_CLEAN_T_FINAL', '1e-3', '1e-3'))
TEMPLATE_TOP_K = int(profile_default('KLDM_WYCKOFF_PPR_TEMPLATE_TOP_K', '3', '6'))
FIXTURE_MAX_TEMPLATES = int(profile_default('KLDM_WYCKOFF_PPR_MAX_TEMPLATES', '12', '20'))
FIXTURE_TEMPLATE_EVAL_LIMIT = int(profile_default('KLDM_WYCKOFF_PPR_TEMPLATE_EVAL_LIMIT', '4', '8'))
FIXTURE_OPT_STEPS = int(profile_default('KLDM_WYCKOFF_PPR_OPT_STEPS', '80', '120'))
FIXTURE_LR = float(profile_default('KLDM_WYCKOFF_PPR_LR', '5e-2', '5e-2'))
EPS_PASS = 1.0e-6
DET_PPR_CLEAN_V0_ZERO = True

FACIT_COMPARE_TAU = float(profile_default('KLDM_WYCKOFF_PPR_FACIT_TAU', '0.25', '0.25'))
FACIT_COMPARE_CORRECTION_STEPS = int(profile_default('KLDM_WYCKOFF_PPR_FACIT_CORRECTION_STEPS', '1', '1'))
MIN_PAIR_DISTANCE_GUARD = float(profile_default('KLDM_WYCKOFF_PPR_MIN_PAIR_DISTANCE_GUARD', '0.5', '0.5'))
SCORE_RATIO_THRESHOLD = float(profile_default('KLDM_WYCKOFF_PPR_SCORE_RATIO_THRESHOLD', '3.0', '3.0'))

DET_PPR_OUTER_STEPS = int(profile_default('KLDM_WYCKOFF_DET_PPR_OUTER_STEPS', '1000', '1000'))
DET_COMPARE_T_VALUES = tuple(float(v.strip()) for v in str(profile_default('KLDM_WYCKOFF_DET_COMPARE_T_VALUES', '0.2,0.3,0.5,0.8', '0.2,0.3,0.5,0.8')).split(',') if v.strip())
DET_PPR_ONCE_T_VALUES = tuple(float(v.strip()) for v in str(profile_default('KLDM_WYCKOFF_DET_PPR_ONCE_T_VALUES', '0.5', '0.5')).split(',') if v.strip())
DET_PPR_EVERY100_AFTER_T_VALUES = tuple(float(v.strip()) for v in str(profile_default('KLDM_WYCKOFF_DET_PPR_EVERY100_AFTER_T_VALUES', '0.5', '0.5')).split(',') if v.strip())
DET_PPR_EVERY_K_STEPS = int(profile_default('KLDM_WYCKOFF_DET_PPR_EVERY_K_STEPS', '200', '200'))
DET_PPR_MEAN_FREE_THRESHOLD = float(profile_default('KLDM_WYCKOFF_DET_PPR_MEAN_FREE_THRESHOLD', '1e-5', '1e-5'))
RUN_SAMPLER_BACKEND_CONSISTENCY = str(profile_default('KLDM_WYCKOFF_RUN_SAMPLER_BACKEND_CONSISTENCY', '0', '0')).strip().lower() not in {'0', 'false', 'no'}

print(f'profile={PROFILE} graphs={GRAPH_IDS} outer_steps={DET_PPR_OUTER_STEPS} baselines=facitKLDM-EM-1000,facitKLDM-PC-1000 primary_baseline=facitKLDM-PC-1000')
print(f'deterministic compare t_values={DET_COMPARE_T_VALUES}')
print(f'deterministic clean-project-renoise once_t={DET_PPR_ONCE_T_VALUES} every{DET_PPR_EVERY_K_STEPS}_after={DET_PPR_EVERY100_AFTER_T_VALUES}')
print(f'clean_estimator_steps={CLEAN_ESTIMATOR_STEPS} clean_t_final={CLEAN_ESTIMATOR_T_FINAL} mean_free_threshold={DET_PPR_MEAN_FREE_THRESHOLD}')
print('This notebook is tuned for deterministic KLDM-Wyckoff clean-project-renoise tests.')




In [ ]:
set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
template_prior = runner._ensure_template_prior()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
FULL_NUM_GRAPHS_IN_BATCH = max(len(full_ptr) - 1, 0)
REQUESTED_GRAPH_IDS = [int(graph_id) for graph_id in GRAPH_IDS]
AVAILABLE_GRAPH_IDS = [graph_id for graph_id in REQUESTED_GRAPH_IDS if 1 <= int(graph_id) <= FULL_NUM_GRAPHS_IN_BATCH]
MISSING_GRAPH_IDS = [graph_id for graph_id in REQUESTED_GRAPH_IDS if int(graph_id) not in AVAILABLE_GRAPH_IDS]
if not AVAILABLE_GRAPH_IDS:
    raise RuntimeError(
        f'No requested graph ids are present in the loaded batch. requested={REQUESTED_GRAPH_IDS} available_range=1..{FULL_NUM_GRAPHS_IN_BATCH}'
    )
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in AVAILABLE_GRAPH_IDS]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    if not selected_items:
        raise RuntimeError(f'Failed to build reduced working batch: selected graph list is empty for requested={AVAILABLE_GRAPH_IDS}')
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.tolist()
NUM_GRAPHS_IN_BATCH = max(len(ptr) - 1, 0)
WORKING_GRAPH_IDS = list(AVAILABLE_GRAPH_IDS)
if int(NUM_GRAPHS_IN_BATCH) != len(WORKING_GRAPH_IDS):
    raise RuntimeError(
        f'Reduced working batch size mismatch: expected={len(WORKING_GRAPH_IDS)} got={NUM_GRAPHS_IN_BATCH}'
    )

sampler_cfgs = {cfg['name']: dict(cfg) for cfg in COMPARE_CFG['samplers']}
facit_cfg = sampler_cfgs.get('FacitKLDM_PC', COMPARE_CFG['samplers'][0])
FIXED_CFG = FixedTemplateVelocityConfig(projector_damping=1.0e-6)

@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_lattice: torch.Tensor
    requested_sg: int


@dataclass
class KLDMState:
    f: torch.Tensor
    v: torch.Tensor
    l: torch.Tensor
    k: torch.Tensor
    h: torch.Tensor
    t: float
    dt: float
    graph_idx0: int
    full_state: Optional[dict[str, Any]] = None
    full_times: Optional[Any] = None


@dataclass
class FixedTemplateFixture:
    case: GraphCase
    state: Any
    theta: torch.Tensor
    tau: torch.Tensor
    assignment: torch.Tensor
    target_k: torch.Tensor
    z_frac: torch.Tensor
    z_l: torch.Tensor
    chart_atomic_numbers: torch.Tensor
    template_labels: str
    template_multiplicities: str
    template_dofs: str
    template_total_dof: int
    requested_sg_match: bool
    composition_match: bool
    projection_objective: float
    candidate_standardization: str


result_tables: dict[str, pd.DataFrame] = {}
result_artifacts: dict[tuple[str, int], dict[str, Any]] = {}
_caches: dict[tuple[Any, ...], Any] = {}
fixtures: dict[int, FixedTemplateFixture] = {}

def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_tensors(graph_idx0: int, *, source=None) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    if source is None:
        pos, l, h = batch.pos, batch.l, batch.atomic_numbers
    else:
        if len(source) == 4:
            pos, _v, l, h = source
        else:
            pos, l, h = source
    return {
        'pos': pos[start:end].detach().clone(),
        'l': l[graph_idx0].detach().clone().reshape(-1),
        'h': h[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=WORKING_GRAPH_IDS) -> list[GraphCase]:
    requested_ids = [int(graph_id) for graph_id in graph_ids]
    out = []
    for graph_idx0, graph_id in enumerate(requested_ids):
        if graph_idx0 >= int(NUM_GRAPHS_IN_BATCH):
            break
        g = graph_tensors(graph_idx0)
        out.append(
            GraphCase(
                graph_id=int(graph_id),
                graph_idx0=int(graph_idx0),
                composition=composition_counter(g['h']),
                atomic_numbers=g['h'],
                gt_frac=g['pos'],
                gt_lattice=g['l'],
                requested_sg=g['sg'],
            )
        )
    return out


GRAPH_CASES = load_test_graphs(AVAILABLE_GRAPH_IDS)
backend_summary = {
    'method': 'deterministic KLDM-Wyckoff clean-project-renoise',
    'short_name': 'detKLDM-Wyckoff-CPR',
    'ppr_relation': 'PPR-inspired x0-space clean-project-renoise approximation; not full xt-space projection-through-denoiser PPR',
    'active_projection_mode': 'clean_x0_project_renoise',
    'future_xt_projection_through_denoiser_reserved': True,
    'future_xt_projection_through_denoiser_active': False,
    'future_xt_projection_reason': 'Reserved placeholder exists, but this notebook intentionally tests the non-differentiable deterministic CPR path only.',
    'clean_estimator_backend': 'deterministic_predictor_clean_estimate: reverse-exp coordinate surrogate + predictor lattice surrogate',
    'clean_estimator_source': PPR_CLEAN_ESTIMATOR_SOURCE,
    'clean_estimator_reason': PPR_CLEAN_ESTIMATOR_REASON,
    'clean_velocity_policy': 'V0_C = 0 for KLDM forward renoise',
    'exact_clean_v0_supported': PPR_EXACT_CLEAN_V0_SUPPORTED,
    'exact_clean_v0_reason': PPR_EXACT_CLEAN_V0_REASON,
    'projection_backend': 'project_clean_to_fixed_wyckoff / project_to_fixed_template_local on fixed Wyckoff chart',
    'constraint_scope': 'fractional-coordinate Wyckoff constraint only',
    'lattice_policy': PPR_LATTICE_POLICY,
    'lattice_policy_reason': PPR_LATTICE_POLICY_REASON,
    'fixed_condition_source': 'batch.space_group + composition/atomic_numbers + fixed template W from oracle structure',
    'oracle_space_group_from_batch': True,
    'oracle_template_selection_from_gt_structure': True,
    'oracle_local_chart_initialized_from_first_projection': True,
    'facit_compare_tau': float(FACIT_COMPARE_TAU),
    'facit_compare_n_steps': int(DET_PPR_OUTER_STEPS),
    'state_backend': 'SamplingCompareRunner/model._prepare_csp_sampling + facit predictor-corrector reverse steps',
    'final_success_metric': 'mean match rate, then mean RMSE against primary baseline facitKLDM-PC-1000; report both facitKLDM-EM-1000 and facitKLDM-PC-1000 baselines',
}
print(f'requested_graphs={REQUESTED_GRAPH_IDS} available_in_loaded_batch={AVAILABLE_GRAPH_IDS} missing_from_loaded_batch={MISSING_GRAPH_IDS}')
print(f'working_batch_num_graphs={NUM_GRAPHS_IN_BATCH} working_graph_ids={WORKING_GRAPH_IDS} full_loaded_batch_num_graphs={FULL_NUM_GRAPHS_IN_BATCH}')
print(f'loaded_graphs={[g.graph_id for g in GRAPH_CASES]} requested_sg={[g.requested_sg for g in GRAPH_CASES]}')
print('backend_summary=', backend_summary)


## Config


In [ ]:
config_summary = pd.DataFrame([{
    'profile': PROFILE,
    'graphs': ','.join(str(g) for g in GRAPH_IDS),
    'outer_steps': int(DET_PPR_OUTER_STEPS),
    'baselines': ['facitKLDM-EM-1000', 'facitKLDM-PC-1000'],
    'primary_baseline': 'facitKLDM-PC-1000',
    'once_t_values': ','.join(f'{t:.1f}' for t in DET_PPR_ONCE_T_VALUES),
    'every100_after_t_values': ','.join(f'{t:.1f}' for t in DET_PPR_EVERY100_AFTER_T_VALUES),
    'every_k_steps': int(DET_PPR_EVERY_K_STEPS),
    'clean_estimator_steps': int(CLEAN_ESTIMATOR_STEPS),
    'clean_t_final': float(CLEAN_ESTIMATOR_T_FINAL),
    'keep_lattice_from_facit_pc': bool(PPR_LATTICE_POLICY == 'keep_reverse_pc_lattice'),
    'clean_v0_zero_for_renoise': bool(DET_PPR_CLEAN_V0_ZERO),
    'hard_skip_mean_free_threshold': float(DET_PPR_MEAN_FREE_THRESHOLD),
}])
result_tables['config_summary'] = config_summary
display(config_summary)


In [ ]:
def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def print_group_pass(df: pd.DataFrame, group_cols) -> None:
    if df.empty:
        print('empty')
        return
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    cols = list(group_cols) + ['PASS', 'status']
    display(df[cols].groupby(group_cols, as_index=False).agg({'PASS': 'all', 'status': 'last'}))


def l_to_k(l: torch.Tensor, graph_case: GraphCase) -> torch.Tensor:
    return _l_to_k(
        l=l.reshape(-1),
        num_atoms=int(graph_case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
    ).to(device=l.device, dtype=l.dtype)


def ensure_l_feature(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        encoded = _encode_lattice_matrix(
            cell_matrix=flat.reshape(3, 3),
            num_atoms=int(num_atoms),
            lattice_transform=runner.lattice_transform,
        )
        return encoded.to(device=l.device, dtype=l.dtype).reshape(-1)
    return flat


def cell_from_l(l: torch.Tensor, num_atoms: int) -> torch.Tensor:
    flat = l.reshape(-1)
    if int(flat.numel()) == 9:
        return flat.reshape(3, 3).detach()
    return _decode_lattice_matrix(
        l=flat,
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).detach()


def volume_from_l(l: torch.Tensor, num_atoms: int) -> float:
    return float(torch.abs(torch.linalg.det(cell_from_l(l, num_atoms))).detach().item())


def min_pair_distance_from_arrays(frac: torch.Tensor, l: torch.Tensor, num_atoms: int) -> float:
    cell = cell_from_l(l, num_atoms).to(device=frac.device, dtype=frac.dtype)
    distances = _periodic_pairwise_distances(frac_coords=frac, cell_matrix=cell)
    return float(distances.min().detach().item()) if distances.numel() else float('inf')


def clone_full_state(full_state: dict[str, Any]) -> dict[str, Any]:
    cloned: dict[str, Any] = {}
    for key, value in full_state.items():
        cloned[key] = value.clone() if torch.is_tensor(value) else value
    return cloned


def _native_reverse_step_full_state(full_state: dict[str, Any], times) -> dict[str, Any]:
    with torch.no_grad():
        preds_curr = full_state['score_network'](
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds_curr['v'],
            index=full_state['node_index'],
        )
        full_state['f_t'], full_state['v_t'] = full_state['sampling_tdm'].reverse_exp_step(
            f_t=full_state['f_t'],
            v_t=full_state['v_t'],
            score_v=score_v,
            index=full_state['node_index'],
            dt=times.dt,
        )
        full_state['l_t'] = runner.model._reverse_lattice_sampling_step(
            t=times.now.lattice,
            x_t=full_state['l_t'],
            pred=preds_curr['l'],
            dt=times.dt,
            num_atoms=batch.num_atoms,
        )
    return full_state


def graph_state_from_full(full_state: dict[str, Any], case: GraphCase, times=None) -> KLDMState:
    start, end = graph_slice(case.graph_idx0)
    f = full_state['f_t'][start:end].detach().clone()
    v = full_state['v_t'][start:end].detach().clone()
    l = full_state['l_t'][case.graph_idx0].detach().clone().reshape(-1)
    h = full_state['a_t'][start:end].detach().clone().to(torch.long)
    k = l_to_k(l, case)
    dt = float(times.dt) if times is not None else 1.0 / max(1, CAPTURE_N_STEPS)
    t = float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item()) if times is not None else float('nan')
    return KLDMState(f=f, v=v, l=l, k=k, h=h, t=t, dt=dt, graph_idx0=case.graph_idx0, full_state=full_state, full_times=times)


def capture_batch_kldm_state(seed=SAMPLE_SEED, n_steps=CAPTURE_N_STEPS, capture_step=CAPTURE_STEP):
    key = ('capture', int(seed), int(n_steps), int(capture_step))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    last_times = None
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        last_times = times
        state = _native_reverse_step_full_state(state, times)
        if step_idx >= int(capture_step):
            break
    _caches[key] = (state, last_times, int(capture_step))
    return _caches[key]


def capture_prepost_reverse_step(seed=SAMPLE_SEED, n_steps=CAPTURE_N_STEPS, target_step=CAPTURE_STEP):
    key = ('prepost', int(seed), int(n_steps), int(target_step))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    pre_state = None
    post_state = None
    step_times = None
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        if step_idx == int(target_step):
            pre_state = clone_full_state(state)
            step_times = times
        state = _native_reverse_step_full_state(state, times)
        if step_idx == int(target_step):
            post_state = clone_full_state(state)
            break
    if pre_state is None or post_state is None or step_times is None:
        raise RuntimeError(f'Could not capture pre/post reverse step target_step={target_step}.')
    _caches[key] = (pre_state, step_times, post_state)
    return _caches[key]


def sample_kldm_state_for_graph_at_step(case: GraphCase, *, capture_step=CAPTURE_STEP, n_steps=CAPTURE_N_STEPS, seed=SAMPLE_SEED) -> KLDMState:
    full_state, times, _step_idx = capture_batch_kldm_state(seed=seed, n_steps=n_steps, capture_step=capture_step)
    return graph_state_from_full(clone_full_state(full_state), case, times=times)


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = (
        (edge_index[0] >= start)
        & (edge_index[0] < end)
        & (edge_index[1] >= start)
        & (edge_index[1] < end)
    )
    local_edge = edge_index[:, mask] - start
    return local_edge.detach().clone()


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(
        pos=pos,
        batch=batch_index,
        num_graphs=1,
        num_atoms=num_atoms,
        edge_node_index=edge_node_index,
    )


def to_algo16_state(state: KLDMState) -> KLDMGraphState:
    return KLDMGraphState(
        f=state.f,
        v=state.v,
        l=ensure_l_feature(state.l, int(state.f.shape[0])),
        h=state.h,
        k=state.k,
        t=state.t,
        dt=state.dt,
        graph_idx0=state.graph_idx0,
    )


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    pred_l_feature = ensure_l_feature(pred_l, int(pred_f.shape[0]))
    base_result = evaluate_csp_reconstruction_kldm(
        pred_f=pred_f,
        pred_l=pred_l_feature,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_lattice,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )
    diag_result = evaluate_csp_reconstruction_kldmplus(
        pred_f=pred_f,
        pred_l=pred_l_feature,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_lattice,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    standardized_frac_rmse = None if diag_result.matcher_diagnostics is None else diag_result.matcher_diagnostics.standardized_frac_rmse
    target_volume = volume_from_l(case.gt_lattice, int(case.gt_frac.shape[0]))
    pred_volume = float(diag_result.volume) if diag_result.volume is not None else float('nan')
    volume_ratio = float(pred_volume / target_volume) if (math.isfinite(pred_volume) and math.isfinite(target_volume) and abs(target_volume) > 1.0e-12) else float('nan')
    return {
        'match': bool(base_result.match),
        'valid': bool(base_result.valid),
        'rmse': base_result.rmse,
        'frac_rmse': diag_result.frac_rmse,
        'standardized_frac_rmse': standardized_frac_rmse,
        'composition_match': diag_result.composition_match,
        'requested_sg_match': diag_result.requested_space_group_match,
        'min_pair_distance': diag_result.min_pair_distance,
        'volume': pred_volume,
        'volume_ratio': volume_ratio,
        'lattice_lengths_mae': diag_result.lattice_lengths_mae,
        'lattice_angles_mae': diag_result.lattice_angles_mae,
        'validity_reason': diag_result.validity_reason,
        'evaluation_source_primary': 'src/kldm/sample_evaluation/evaluate_csp_reconstruction',
        'evaluation_source_diagnostics': 'src/kldmPlus/sample_evaluation/evaluate_csp_reconstruction',
    }



def score_norms_from_arrays(case: GraphCase, base_state: KLDMState, *, f: torch.Tensor, v: torch.Tensor, l: torch.Tensor, h: torch.Tensor | None = None) -> dict[str, Any]:
    if not np.isfinite(float(base_state.t)):
        return {
            'finite_ok': False,
            'score_eval_success': False,
            'score_v_norm': float('nan'),
            'pred_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'score_norm_max': float('nan'),
            'baseline_valid': False,
            'status': 'UNKNOWN',
            'error_type': 'ScoreTimeUnavailable',
            'error_message': 'Non-finite graph time; final sampled states cannot be scored with this helper.',
        }
    try:
        h_use = (base_state.h if h is None else h).reshape(-1).to(device=f.device, dtype=torch.long)
        f_use = f.reshape(-1, 3)
        v_use = v.reshape(-1, 3)
        batch_view = make_single_graph_batch_view(case, ref_tensor=f_use)
        node_index = batch_view.batch.reshape(-1).to(torch.long)
        edge_node_index = batch_view.edge_node_index.to(torch.long)
        l_feature = ensure_l_feature(l, int(f_use.shape[0]))
        l_graph = l_feature.reshape(1, -1).to(device=f_use.device, dtype=f_use.dtype)
        t_graph = torch.as_tensor([float(base_state.t)], device=f_use.device, dtype=f_use.dtype)
        t_nodes = torch.full((f_use.shape[0],), float(base_state.t), device=f_use.device, dtype=f_use.dtype)
        with torch.no_grad():
            preds = runner.model.score_network(
                t=t_graph,
                pos=f_use,
                v=v_use,
                h=h_use,
                l=l_graph,
                node_index=node_index,
                edge_node_index=edge_node_index,
            )
            score_v = runner.model.tdm.reconstruct_full_reverse_velocity_score(
                t=t_nodes,
                v_t=v_use,
                pred_v=preds['v'],
                index=node_index,
            )
        finite_ok = bool(
            torch.isfinite(preds['v']).all().item()
            and torch.isfinite(preds['l']).all().item()
            and torch.isfinite(score_v).all().item()
        )
        score_v_norm = float(torch.linalg.norm(score_v.reshape(-1)).detach().item()) if finite_ok else float('nan')
        pred_l_norm = float(torch.linalg.norm(preds['l'].reshape(-1)).detach().item()) if finite_ok else float('nan')
        pred_v_norm = float(torch.linalg.norm(preds['v'].reshape(-1)).detach().item()) if finite_ok else float('nan')
        score_norm_max = max(score_v_norm, pred_l_norm, pred_v_norm) if finite_ok else float('nan')
        baseline_valid = bool(finite_ok and math.isfinite(score_norm_max) and score_norm_max > 1.0e-12)
        return {
            'finite_ok': finite_ok,
            'score_eval_success': finite_ok,
            'score_v_norm': score_v_norm,
            'pred_v_norm': pred_v_norm,
            'pred_l_norm': pred_l_norm,
            'score_norm_max': score_norm_max,
            'baseline_valid': baseline_valid,
            'status': 'OK' if finite_ok else 'ERROR',
            'error_type': None,
            'error_message': None,
        }
    except Exception as exc:
        return {
            'finite_ok': False,
            'score_eval_success': False,
            'score_v_norm': float('nan'),
            'pred_v_norm': float('nan'),
            'pred_l_norm': float('nan'),
            'score_norm_max': float('nan'),
            'baseline_valid': False,
            'status': 'ERROR',
            'error_type': type(exc).__name__,
            'error_message': str(exc),
        }


def robust_score_ratio(before: dict[str, Any], after: dict[str, Any], eps: float = 1.0e-12) -> tuple[float, bool]:
    before_val = float(before.get('score_norm_max', float('nan')))
    after_val = float(after.get('score_norm_max', float('nan')))
    baseline_valid = bool(before.get('baseline_valid', False))
    after_valid = bool(after.get('finite_ok', False) and math.isfinite(after_val))
    if not baseline_valid or not after_valid:
        return float('nan'), False
    return after_val / max(before_val, eps), True


def quotient_idempotence_proxy(source_frac: torch.Tensor, target_frac: torch.Tensor, atomic_numbers: torch.Tensor) -> dict[str, Any]:
    source_frac = source_frac.reshape(-1, 3)
    target_frac = target_frac.reshape(-1, 3)
    atomic_numbers = atomic_numbers.reshape(-1).to(torch.long)
    if int(source_frac.shape[0]) != int(target_frac.shape[0]) or int(source_frac.shape[0]) != int(atomic_numbers.shape[0]):
        return {'distance': float('nan'), 'translation': None, 'assignment': None}
    best = {'distance': float('inf'), 'translation': None, 'assignment': None}
    n = int(source_frac.shape[0])
    for i in range(n):
        zi = int(atomic_numbers[i].item())
        for j in range(n):
            if int(atomic_numbers[j].item()) != zi:
                continue
            tau = torch.remainder(target_frac[j] - source_frac[i], 1.0)
            shifted = torch.remainder(source_frac + tau.unsqueeze(0), 1.0)
            assignment = _species_assignment_indices(
                source_frac=shifted,
                source_atomic_numbers=atomic_numbers,
                target_frac=target_frac,
                target_atomic_numbers=atomic_numbers,
            ).to(device=source_frac.device, dtype=torch.long)
            matched = target_frac[assignment]
            dist = float(torch.linalg.norm(wrap_residual(shifted, matched).reshape(-1)).detach().item())
            if dist < best['distance']:
                best = {
                    'distance': dist,
                    'translation': tau.detach().cpu().tolist(),
                    'assignment': assignment.detach().cpu().tolist(),
                }
    return best


def capture_batch_pre_state_at_t_threshold(
    seed=SAMPLE_SEED,
    n_steps=CAPTURE_N_STEPS,
    t_threshold=0.70,
    sampler_kind: str = 'facit_pc',
    tau: float = FACIT_COMPARE_TAU,
    n_correction_steps: int = FACIT_COMPARE_CORRECTION_STEPS,
):
    key = ('capture_t_pre', int(seed), int(n_steps), float(t_threshold), str(sampler_kind), float(tau), int(n_correction_steps))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1e-3)),
    )
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        t_now = float(times.now.graph.reshape(-1)[0].detach().item())
        if t_now <= float(t_threshold):
            _caches[key] = (clone_full_state(state), times, int(step_idx))
            return _caches[key]
        state = _benchmark_sampler_step_full_state(
            state,
            times,
            sampler_kind=str(sampler_kind),
            tau=float(tau),
            n_correction_steps=int(n_correction_steps),
        )
    raise RuntimeError(f'Could not capture pre-state at t_threshold={t_threshold}.')


def sample_kldm_state_for_graph_at_t_threshold(
    case: GraphCase,
    *,
    t_threshold: float,
    n_steps=CAPTURE_N_STEPS,
    seed=SAMPLE_SEED,
    sampler_kind: str = 'facit_pc',
    tau: float = FACIT_COMPARE_TAU,
    n_correction_steps: int = FACIT_COMPARE_CORRECTION_STEPS,
) -> KLDMState:
    full_state, times, _step_idx = capture_batch_pre_state_at_t_threshold(
        seed=seed,
        n_steps=n_steps,
        t_threshold=t_threshold,
        sampler_kind=str(sampler_kind),
        tau=float(tau),
        n_correction_steps=int(n_correction_steps),
    )
    return graph_state_from_full(clone_full_state(full_state), case, times=times)


def write_graph_state_into_full_state(full_state: dict[str, Any], case: GraphCase, *, f: torch.Tensor, v: torch.Tensor, l: torch.Tensor, h: torch.Tensor | None = None) -> None:
    start, end = graph_slice(case.graph_idx0)
    full_state['f_t'][start:end] = f.reshape(end - start, 3).to(device=full_state['f_t'].device, dtype=full_state['f_t'].dtype)
    full_state['v_t'][start:end] = v.reshape(end - start, 3).to(device=full_state['v_t'].device, dtype=full_state['v_t'].dtype)
    l_feature = ensure_l_feature(l, int(end - start)).reshape_as(full_state['l_t'][case.graph_idx0])
    full_state['l_t'][case.graph_idx0] = l_feature.to(device=full_state['l_t'].device, dtype=full_state['l_t'].dtype)
    if h is not None:
        full_state['a_t'][start:end] = h.reshape(end - start).to(device=full_state['a_t'].device, dtype=full_state['a_t'].dtype)


def velocity_norm_value(v: torch.Tensor) -> float:
    return float(torch.linalg.norm(v.reshape(-1)).detach().item())


def sample_facit_pc_full_batch(seed=SAMPLE_SEED, n_steps=CAPTURE_N_STEPS, tau=FACIT_COMPARE_TAU, n_correction_steps=FACIT_COMPARE_CORRECTION_STEPS):
    key = ('facit_pc_full', int(seed), int(n_steps), float(tau), int(n_correction_steps), int(NUM_GRAPHS_IN_BATCH), tuple(int(x) for x in batch.num_atoms.tolist()))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    with torch.no_grad():
        f_t, v_t, l_t, a_t = runner.model.sample_CSP_algorithm4_facit(
            n_steps=int(n_steps),
            batch=batch,
            t_start=float(facit_cfg.get('t_start', 1.0)),
            t_final=float(facit_cfg.get('t_final', 1.0e-3)),
            tau=float(tau),
            n_correction_steps=int(n_correction_steps),
        )
    _caches[key] = (f_t.detach().clone(), v_t.detach().clone(), l_t.detach().clone(), a_t.detach().clone())
    return _caches[key]


def facit_graph_state(case: GraphCase, *, tau: float = FACIT_COMPARE_TAU, n_steps: int = CAPTURE_N_STEPS, seed: int = SAMPLE_SEED) -> KLDMState:
    f_t, v_t, l_t, a_t = sample_facit_pc_full_batch(seed=seed, n_steps=n_steps, tau=tau, n_correction_steps=FACIT_COMPARE_CORRECTION_STEPS)
    g = graph_tensors(case.graph_idx0, source=(f_t, v_t, l_t, a_t))
    return KLDMState(
        f=g['pos'],
        v=v_t[graph_slice(case.graph_idx0)[0]:graph_slice(case.graph_idx0)[1]].detach().clone(),
        l=g['l'],
        k=l_to_k(g['l'], case),
        h=g['h'],
        t=float('nan'),
        dt=1.0 / max(int(n_steps), 1),
        graph_idx0=case.graph_idx0,
        full_state=None,
        full_times=None,
    )


def projection_shock_to_fixture(case: GraphCase, state: KLDMState, *, fixture_override=None) -> dict[str, Any]:
    fixture = fixtures[case.graph_id] if fixture_override is None else fixture_override
    use_clean_estimate = state.full_state is not None and np.isfinite(float(state.t))
    if use_clean_estimate:
        clean = gate_clean_estimate(case, state)
        reference_frac = clean.f0_hat
    else:
        clean = None
        reference_frac = state.f
    projection = project_clean_to_fixed_wyckoff(
        f0_hat=reference_frac,
        atomic_numbers=state.h,
        template_state=fixture.state,
        target_k=fixture.target_k,
        tau0=fixture.tau,
        theta0=fixture.theta,
        fixed_assignment=fixture.assignment,
        config=FIXED_CFG,
        reference_frac=reference_frac,
    )
    shock = projection_distance_to_fixture(projection.z_f, reference_frac)
    return {'clean': clean, 'projection': projection, 'shock': float(shock), 'reference_frac': reference_frac, 'used_clean_estimate': bool(use_clean_estimate)}


def _facit_pc_step_full_state(full_state: dict[str, Any], times, *, tau: float, n_correction_steps: int) -> dict[str, Any]:
    with torch.no_grad():
        for _ in range(max(int(n_correction_steps), 1)):
            preds_corr = runner.model.score_network(
                t=times.now.graph,
                pos=full_state['f_t'],
                v=full_state['v_t'],
                h=full_state['a_t'],
                l=full_state['l_t'],
                node_index=full_state['node_index'],
                edge_node_index=full_state['edge_node_index'],
            )
            full_state['f_t'], full_state['v_t'] = full_state['sampling_tdm'].reverse_step_corrector(
                t=times.now.nodes,
                f_t=full_state['f_t'],
                v_t=full_state['v_t'],
                pred_v=preds_corr['v'],
                dt=times.dt,
                index=full_state['node_index'],
                tau=float(tau),
            )
            full_state['l_t'] = full_state['sampling_diffusion_l'].reverse_step_corrector(
                t=times.now.lattice,
                x_t=full_state['l_t'],
                pred=preds_corr['l'],
                tau=float(tau),
            )
        preds_pred = runner.model.score_network(
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        full_state['f_t'], full_state['v_t'] = full_state['sampling_tdm'].reverse_step_predictor(
            t=times.now.nodes,
            f_t=full_state['f_t'],
            v_t=full_state['v_t'],
            pred_v=preds_pred['v'],
            index=full_state['node_index'],
            dt=times.dt,
        )
        full_state['l_t'] = full_state['sampling_diffusion_l'].reverse_step_predictor(
            t=times.now.lattice,
            x_t=full_state['l_t'],
            pred=preds_pred['l'],
            dt=times.dt,
        )
    return full_state


def _facit_em_step_full_state(full_state: dict[str, Any], times) -> dict[str, Any]:
    with torch.no_grad():
        preds_curr = runner.model.score_network(
            t=times.now.graph,
            pos=full_state['f_t'],
            v=full_state['v_t'],
            h=full_state['a_t'],
            l=full_state['l_t'],
            node_index=full_state['node_index'],
            edge_node_index=full_state['edge_node_index'],
        )
        score_v = full_state['sampling_tdm'].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state['v_t'],
            pred_v=preds_curr['v'],
            index=full_state['node_index'],
        )
        full_state['f_t'], full_state['v_t'] = full_state['sampling_tdm'].reverse_exp_step(
            f_t=full_state['f_t'],
            v_t=full_state['v_t'],
            score_v=score_v,
            index=full_state['node_index'],
            dt=times.dt,
        )
        full_state['l_t'] = full_state['sampling_diffusion_l'].reverse_step(
            t=times.now.lattice,
            x_t=full_state['l_t'],
            pred=preds_curr['l'],
            dt=times.dt,
        )
    return full_state


def _benchmark_sampler_step_full_state(full_state: dict[str, Any], times, *, sampler_kind: str, tau: float, n_correction_steps: int) -> dict[str, Any]:
    sampler_kind = str(sampler_kind).strip().lower()
    if sampler_kind == 'facit_pc':
        return _facit_pc_step_full_state(full_state, times, tau=float(tau), n_correction_steps=int(n_correction_steps))
    if sampler_kind == 'facit_em':
        return _facit_em_step_full_state(full_state, times)
    if sampler_kind == 'native_em':
        return _native_reverse_step_full_state(full_state, times)
    raise ValueError(f'Unsupported sampler_kind={sampler_kind!r}')


def guarded_final_local_projection(case: GraphCase, state: KLDMState, *, template_state, target_k, tau0, theta0, fixed_assignment) -> dict[str, Any]:
    eval_unproj = evaluate_arrays(case, state.f, state.l, state.h)
    projection = project_clean_to_fixed_wyckoff(
        f0_hat=state.f,
        atomic_numbers=state.h,
        template_state=template_state,
        target_k=target_k,
        tau0=tau0,
        theta0=theta0,
        fixed_assignment=fixed_assignment,
        config=FIXED_CFG,
        reference_frac=state.f,
    )
    shock = projection_distance_to_fixture(projection.z_f, state.f)
    eval_proj = evaluate_arrays(case, projection.z_f, state.l, state.h)
    reasons = []
    if not bool(projection.projection_success):
        reasons.append('projection_failed')
    if not bool(eval_proj['valid']):
        reasons.append('invalid_structure')
    min_dist = float(eval_proj['min_pair_distance']) if eval_proj['min_pair_distance'] is not None else float('nan')
    if not math.isfinite(min_dist) or min_dist <= float(MIN_PAIR_DISTANCE_GUARD):
        reasons.append('close_contact')
    rmse_unproj = eval_unproj['rmse']
    rmse_proj = eval_proj['rmse']
    match_improves = bool(eval_proj['match']) and not bool(eval_unproj['match'])
    rmse_not_worse = (
        rmse_proj is not None and rmse_unproj is not None and math.isfinite(float(rmse_proj)) and math.isfinite(float(rmse_unproj)) and float(rmse_proj) <= float(rmse_unproj) + 1.0e-12
    )
    if not (match_improves or rmse_not_worse or (rmse_unproj is None and eval_proj['match'])):
        reasons.append('no_match_or_rmse_gain')
    accepted = len(reasons) == 0
    return {
        'projection': projection,
        'shock': float(shock),
        'accepted': bool(accepted),
        'rejected_reason': ';'.join(reasons) if reasons else None,
        'eval_unproj': eval_unproj,
        'eval_proj': eval_proj,
        'final_frac': projection.z_f if accepted else state.f,
    }


def debug_mixed_clean_estimate(
    *,
    model,
    batch,
    state: KLDMGraphState,
    node_index: torch.Tensor,
    edge_node_index: torch.Tensor,
    n_steps: int,
    t_start: float,
    t_final: float = 1.0e-3,
) -> KLDMCleanEstimate:
    exp_clean = deterministic_predictor_clean_estimate(
        model=model,
        batch=batch,
        state=state,
        node_index=node_index,
        edge_node_index=edge_node_index,
        n_steps=int(n_steps),
        t_start=float(t_start),
        t_final=float(t_final),
    )
    score_network = model.score_network
    restore_training = score_network.training
    score_network.eval()
    try:
        grid = sampling_grid(
            batch=batch,
            n_steps=max(int(n_steps), 1),
            t_start=float(t_start),
            t_final=float(t_final),
        )
        f_t = state.f.detach().clone()
        v_t = state.v.detach().clone()
        l_t = state.l.detach().clone().reshape(1, -1)
        h_t = state.h.detach().clone()
        for times in iter_sampling_times(batch=batch, grid=grid):
            for _ in range(max(int(FACIT_COMPARE_CORRECTION_STEPS), 1)):
                preds_corr = model.score_network(
                    t=times.now.graph,
                    pos=f_t,
                    v=v_t,
                    h=h_t,
                    l=l_t,
                    node_index=node_index,
                    edge_node_index=edge_node_index,
                )
                f_t, v_t = model.tdm.reverse_step_corrector(
                    t=times.now.nodes,
                    f_t=f_t,
                    v_t=v_t,
                    pred_v=preds_corr['v'],
                    dt=times.dt,
                    index=node_index,
                    tau=float(FACIT_COMPARE_TAU),
                )
                l_t = model.diffusion_l.reverse_step_corrector(
                    t=times.now.lattice,
                    x_t=l_t,
                    pred=preds_corr['l'],
                    tau=float(FACIT_COMPARE_TAU),
                )
            preds_pred = model.score_network(
                t=times.now.graph,
                pos=f_t,
                v=v_t,
                h=h_t,
                l=l_t,
                node_index=node_index,
                edge_node_index=edge_node_index,
            )
            f_t, v_t = model.tdm.reverse_step_predictor(
                t=times.now.nodes,
                f_t=f_t,
                v_t=v_t,
                pred_v=preds_pred['v'],
                index=node_index,
                dt=times.dt,
            )
            l_t = model.diffusion_l.reverse_step_predictor(
                t=times.now.lattice,
                x_t=l_t,
                pred=preds_pred['l'],
                dt=times.dt,
                num_atoms=batch.num_atoms,
            )
        return KLDMCleanEstimate(
            f0_hat=exp_clean.f0_hat.detach().clone(),
            v0_hat=exp_clean.v0_hat.detach().clone(),
            l0_hat=l_t[0].detach().clone(),
            steps=int(n_steps),
            estimator_mode='exp_coords__facit_pc_lattice_1000step_debug',
        )
    finally:
        if restore_training:
            score_network.train()


def aggregate_benchmark(df_samples: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method, dfm in df_samples.groupby('method', sort=False):
        mean_match_rate = float(dfm['match'].astype(float).mean()) if not dfm.empty else float('nan')
        mean_rmse = float(dfm['RMSE'].dropna().astype(float).mean()) if 'RMSE' in dfm.columns and dfm['RMSE'].notna().any() else float('nan')
        median_rmse = float(dfm['RMSE'].dropna().astype(float).median()) if 'RMSE' in dfm.columns and dfm['RMSE'].notna().any() else float('nan')
        mean_frac_rmse = float(dfm['frac_RMSE'].dropna().astype(float).mean()) if 'frac_RMSE' in dfm.columns and dfm['frac_RMSE'].notna().any() else float('nan')
        valid_rate = float(dfm['valid_structure'].astype(float).mean()) if not dfm.empty else float('nan')
        close_contact_rate = float(dfm['close_contact'].astype(float).mean()) if not dfm.empty else float('nan')
        mean_projection_shock = float(dfm['projection_shock'].dropna().astype(float).mean()) if 'projection_shock' in dfm.columns and dfm['projection_shock'].notna().any() else float('nan')
        runtime_mean = float(dfm['runtime'].astype(float).mean()) if 'runtime' in dfm.columns else float('nan')
        accepted_hits_mean = float(dfm['accepted_hits'].astype(float).mean()) if 'accepted_hits' in dfm.columns else float('nan')
        accepted_hits_rate = float((dfm['accepted_hits'].astype(float) > 0.0).mean()) if 'accepted_hits' in dfm.columns else float('nan')
        rows.append({
            'method': method,
            'num_samples': int(len(dfm)),
            'num_graphs': int(len(dfm)),
            'mean_match_rate': mean_match_rate,
            'mean_RMSE': mean_rmse,
            'median_RMSE': median_rmse,
            'mean_frac_RMSE': mean_frac_rmse,
            'valid_rate': valid_rate,
            'close_contact_rate': close_contact_rate,
            'mean_projection_shock': mean_projection_shock,
            'runtime_mean': runtime_mean,
            'runtime': runtime_mean,
            'accepted_hits_mean': accepted_hits_mean,
            'accepted_hits_rate': accepted_hits_rate,
        })
    agg = pd.DataFrame(rows)
    if agg.empty:
        return agg
    primary_baseline_method = globals().get('PRIMARY_BASELINE_METHOD', 'facitKLDM-PC-1000')
    baseline = agg.loc[agg['method'] == primary_baseline_method]
    if baseline.empty:
        agg['beats_baseline_match'] = False
        agg['beats_baseline_RMSE'] = False
        agg['beats_baseline_overall'] = False
        agg['primary_baseline_method'] = primary_baseline_method
        return agg
    base_match = float(baseline['mean_match_rate'].iloc[0])
    base_rmse = float(baseline['mean_RMSE'].iloc[0])
    agg['primary_baseline_method'] = primary_baseline_method
    agg['beats_baseline_match'] = agg['mean_match_rate'] > base_match + 1.0e-12
    agg['beats_baseline_RMSE'] = np.where(
        np.isfinite(agg['mean_RMSE']) & np.isfinite(base_rmse),
        agg['mean_RMSE'] < base_rmse - 1.0e-12,
        False,
    )
    agg['beats_baseline_overall'] = agg['beats_baseline_match'] | ((np.abs(agg['mean_match_rate'] - base_match) <= 1.0e-12) & agg['beats_baseline_RMSE'])
    agg['beats_baseline'] = agg['beats_baseline_overall']
    return agg


def perturb_frac(frac: torch.Tensor, scale: float, seed: int = 0) -> torch.Tensor:
    if float(scale) <= 0.0:
        return frac.detach().clone()
    gen = torch.Generator(device=frac.device)
    gen.manual_seed(int(seed))
    noise = (torch.rand(frac.shape, generator=gen, device=frac.device, dtype=frac.dtype) - 0.5) * (2.0 * float(scale))
    return torch.remainder(frac + noise, 1.0)


def projection_shock_to_fixture(case: GraphCase, state: KLDMState, *, fixture_override=None) -> dict[str, Any]:
    fixture = fixtures[case.graph_id] if fixture_override is None else fixture_override
    clean = gate_clean_estimate(case, state)
    projection = project_clean_to_fixed_wyckoff(
        f0_hat=clean.f0_hat,
        atomic_numbers=state.h,
        template_state=fixture.state,
        target_k=fixture.target_k,
        tau0=fixture.tau,
        theta0=fixture.theta,
        fixed_assignment=fixture.assignment,
        config=FIXED_CFG,
        reference_frac=clean.f0_hat,
    )
    shock = projection_distance_to_fixture(projection.z_f, clean.f0_hat)
    return {'clean': clean, 'projection': projection, 'shock': float(shock)}


In [ ]:
def template_labels(template) -> str:
    return ','.join(f"{site.atomic_number}@{site.label}" for site in template.site_templates)


def template_multiplicities(template) -> str:
    return ','.join(str(int(site.multiplicity)) for site in template.site_templates)


def template_dofs(template) -> str:
    return ','.join(str(int(site.dof)) for site in template.site_templates)


def _pcs_state_chart_target(state0) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, str]:
    chart_frac = state0.target_frac if state0.target_frac is not None else state0.anchor_frac
    chart_atomic_numbers = state0.target_atomic_numbers if state0.target_atomic_numbers is not None else state0.anchor_atomic_numbers
    chart_cell = state0.target_cell if state0.target_cell is not None else state0.anchor_cell
    chart_k = state0.target_k if state0.target_k is not None else state0.anchor_k
    target_repr = str(state0.target_representation_name or state0.anchor_representation_name or 'standardized')
    if chart_frac is None or chart_atomic_numbers is None or chart_cell is None or chart_k is None:
        raise RuntimeError('PCS state is missing target/anchor chart tensors.')
    return chart_frac.detach().clone(), chart_atomic_numbers.detach().clone().to(torch.long), chart_cell.detach().clone(), chart_k.detach().clone(), target_repr


def validate_projection(case: GraphCase, projection) -> Any:
    if hasattr(projection, 'raw'):
        frac_coords = projection.z_f if hasattr(projection, 'z_f') else projection.z_frac
        atomic_numbers = projection.raw.atomic_numbers_chart
        cell_matrix = projection.raw.cell
    else:
        frac_coords = projection.z_frac if hasattr(projection, 'z_frac') else projection.frac_coords_chart
        atomic_numbers = projection.raw.atomic_numbers_chart if hasattr(projection, 'raw') else projection.atomic_numbers_chart
        cell_matrix = projection.raw.cell if hasattr(projection, 'raw') else projection.cell
    structure = _build_vanilla_structure(
        frac_coords=frac_coords,
        atomic_numbers=atomic_numbers,
        cell_matrix=cell_matrix,
    )
    return validate_requested_space_group(
        structure=structure,
        requested_space_group=case.requested_sg,
        expected_atomic_numbers=case.atomic_numbers,
    )


def build_fixed_template_fixture(case: GraphCase) -> FixedTemplateFixture:
    num_atoms = int(case.gt_frac.shape[0])
    target_cell = cell_from_l(case.gt_lattice, num_atoms).to(device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    oracle_structure = _build_vanilla_structure(
        frac_coords=case.gt_frac,
        atomic_numbers=case.atomic_numbers,
        cell_matrix=target_cell,
    )

    def _select_candidates(standardization: str):
        return select_requested_template_states(
            frac_coords=case.gt_frac,
            atomic_numbers=case.atomic_numbers,
            cell_matrix=target_cell,
            space_group_number=case.requested_sg,
            standardization=standardization,
            symprec=1e-2,
            angle_tolerance=5.0,
            max_templates=max(FIXTURE_MAX_TEMPLATES, 16),
            template_eval_limit=max(FIXTURE_TEMPLATE_EVAL_LIMIT, 6),
            optimization_steps=FIXTURE_OPT_STEPS,
            learning_rate=FIXTURE_LR,
            coord_weight=1.0,
            lattice_weight=0.0,
            pairdist_weight=0.0,
            steric_weight=0.0,
            volume_weight=0.0,
            k6_weight=0.0,
            freeze_lattice_free_vars=True,
            quick_templates=False,
            top_k=max(TEMPLATE_TOP_K, 6),
            template_prior=template_prior,
            template_prior_weight=0.0,
            oracle_reference_structure=oracle_structure,
            oracle_fit_structure=oracle_structure,
        )

    candidate_entries = []
    selection_errors = []
    for standardization in ['conventional', 'primitive']:
        try:
            states = _select_candidates(standardization)
        except Exception as exc:
            selection_errors.append((standardization, exc))
            continue
        for state0 in states:
            chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr = _pcs_state_chart_target(state0)
            candidate_entries.append({
                'standardization': standardization,
                'state': state0,
                'chart_frac': chart_frac,
                'chart_atomic_numbers': chart_atomic_numbers,
                'chart_cell': chart_cell,
                'chart_k': chart_k,
                'chart_num_atoms': int(chart_atomic_numbers.shape[0]),
                'same_count': int(chart_atomic_numbers.shape[0]) == num_atoms,
                'target_repr': target_repr,
            })

    if not candidate_entries:
        lines = [f"{std}:{type(exc).__name__}:{exc}" for std, exc in selection_errors]
        raise RuntimeError(f'No fixed-template candidates selected for graph={case.graph_id}. selection_errors={lines}')

    same_count = [item for item in candidate_entries if item['same_count']]
    if not same_count:
        atoms = sorted({int(item['chart_num_atoms']) for item in candidate_entries})
        multiplicity = int(conventional_cell_multiplicity(case.requested_sg))
        conventional_atoms = int(num_atoms * multiplicity)
        raise RuntimeError(
            'TEMPLATE_COUNT_MISMATCH '
            f'graph={case.graph_id} requested_sg={case.requested_sg} graph_atoms={num_atoms} '
            f'conventional_multiplicity={multiplicity} requested_conventional_atoms={conventional_atoms} '
            f'candidate_atoms={atoms} selector_uses_requested_conventional_composition=True'
        )

    same_count.sort(key=lambda item: (0 if item['standardization'] == 'conventional' else 1, float(item['state'].objective)))
    errors = []
    best_payload = None
    for item in same_count:
        state0 = item['state']
        tau0 = torch.zeros(1, 3, device=item['chart_frac'].device, dtype=item['chart_frac'].dtype)
        fixed_assignment = state0.fixed_target_assignment
        if fixed_assignment is None:
            fixed_assignment = state0.anchor_assignment
        try:
            projection = project_to_fixed_template_local(
                f_frac=item['chart_frac'],
                atomic_numbers=item['chart_atomic_numbers'],
                template_state=state0,
                target_k=item['chart_k'],
                tau0=tau0,
                theta0=state0.free_vars,
                fixed_assignment=fixed_assignment,
                config=FIXED_CFG,
            )
            validation = validate_projection(case, projection)
            coord_residual = float(torch.linalg.norm(wrap_residual(item['chart_frac'], projection.z_frac).reshape(-1)).detach().item())
            score = (
                0 if validation.requested_space_group_match else 1,
                0 if validation.composition_match else 1,
                0 if item['standardization'] == 'conventional' else 1,
                coord_residual,
                float(projection.objective),
            )
            fixture = FixedTemplateFixture(
                case=case,
                state=projection.raw.state,
                theta=projection.theta.detach().clone(),
                tau=projection.tau.detach().clone(),
                assignment=projection.assignment.detach().clone(),
                target_k=item['chart_k'].detach().clone(),
                z_frac=projection.z_frac.detach().clone(),
                z_l=projection.z_l.detach().clone(),
                chart_atomic_numbers=item['chart_atomic_numbers'].detach().clone(),
                template_labels=template_labels(projection.raw.state.template),
                template_multiplicities=template_multiplicities(projection.raw.state.template),
                template_dofs=template_dofs(projection.raw.state.template),
                template_total_dof=int(projection.raw.state.template.total_free_dims),
                requested_sg_match=bool(validation.requested_space_group_match),
                composition_match=bool(validation.composition_match),
                projection_objective=float(projection.objective),
                candidate_standardization=item['standardization'],
            )
            if best_payload is None or score < best_payload[0]:
                best_payload = (score, fixture)
        except Exception as exc:
            errors.append(f"std={item['standardization']} {type(exc).__name__}:{exc}")

    if best_payload is None:
        raise RuntimeError(f'Could not build an oracle-W fixed template for graph={case.graph_id}. errors={errors[:8]}')

    return best_payload[1]


## Phase 0 — Template Compatibility And Projector Idempotence


In [ ]:
rows = []
fixtures = {}
ACTIVE_CASES = []
for case in GRAPH_CASES:
    try:
        fixture = build_fixed_template_fixture(case)
        fixtures[case.graph_id] = fixture
        ACTIVE_CASES.append(case)
        proj = project_clean_to_fixed_wyckoff(
            f0_hat=fixture.z_frac,
            atomic_numbers=fixture.chart_atomic_numbers,
            template_state=fixture.state,
            target_k=fixture.target_k,
            tau0=fixture.tau,
            theta0=fixture.theta,
            fixed_assignment=fixture.assignment,
            config=FIXED_CFG,
            reference_frac=fixture.z_frac,
        )
        pass_flag = bool(
            fixture.requested_sg_match
            and fixture.composition_match
            and proj.projection_success
            and float(proj.idempotence_error) < 1.0e-5
            and (not proj.assignment_changed)
            and (not proj.branch_changed)
        )
        rows.append({
            'test': 'template_projection_sanity',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'num_atoms_graph': int(case.gt_frac.shape[0]),
            'num_atoms_template': int(fixture.chart_atomic_numbers.shape[0]),
            'template_labels': fixture.template_labels,
            'theta_dim': int(fixture.theta.numel()),
            'composition_matches_template': bool(fixture.composition_match),
            'requested_sg_matches_template': bool(fixture.requested_sg_match),
            'idempotence_error': float(proj.idempotence_error),
            'projection_objective': float(proj.objective),
            'assignment_changed': bool(proj.assignment_changed),
            'branch_changed': bool(proj.branch_changed),
            'PASS': pass_flag,
            'status': 'PASS' if pass_flag else 'FAIL',
        })
    except Exception as exc:
        rows.append(error_row('template_projection_sanity', case.graph_id, exc, 'TEMPLATE_OR_PROJECTION_SANITY_ERROR', space_group=case.requested_sg))

df_template_projection_sanity = dataframe_result('template_projection_sanity', rows)
display(df_template_projection_sanity)
print('active_cases=', [case.graph_id for case in ACTIVE_CASES])


In [ ]:
def projection_distance_to_fixture(z_f: torch.Tensor, reference_f: torch.Tensor) -> float:
    return float(torch.linalg.norm(wrap_residual(reference_f, z_f).reshape(-1)).detach().item())


def gate_clean_estimate(case: GraphCase, state: KLDMState):
    batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
    return deterministic_predictor_clean_estimate(
        model=runner.model,
        batch=batch_view,
        state=to_algo16_state(state),
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        n_steps=CLEAN_ESTIMATOR_STEPS,
        t_start=float(state.t),
        t_final=float(CLEAN_ESTIMATOR_T_FINAL),
    )


def gate_projection(case: GraphCase, state: KLDMState, clean_estimate) -> Any:
    fixture = fixtures[case.graph_id]
    return project_clean_to_fixed_wyckoff(
        f0_hat=clean_estimate.f0_hat,
        atomic_numbers=state.h,
        template_state=fixture.state,
        target_k=fixture.target_k,
        tau0=fixture.tau,
        theta0=fixture.theta,
        fixed_assignment=fixture.assignment,
        config=FIXED_CFG,
        reference_frac=clean_estimate.f0_hat,
    )


In [ ]:

PRIMARY_BASELINE_METHOD = 'facitKLDM-PC-1000'
DETERMINISTIC_ESTIMATOR_SPECS = [
    ('deterministic_reverse_predictor', 'Deterministic reverse-exp predictor'),
    ('pf_ode', 'PF-ODE clean estimator'),
    ('facit_pc_mixed_debug', 'Basic PC-style deterministic surrogate'),
]
BEST_DETERMINISTIC_ESTIMATOR_MODE = 'pf_ode'

BASELINE_METHOD_SPECS = [
    ('facitKLDM-EM-1000', 'baseline', None, 'facit_em'),
    ('facitKLDM-PC-1000', 'baseline', None, 'facit_pc'),
]

SINGLE_HIT_METHOD_SPECS = [
    ('detPPR-correctW-once-t0.5', 'det_ppr', 'once_t0.5', 'facit_pc'),
    ('detPPR-wrongSG-once-t0.5', 'det_ppr_wrong_space_group_W', 'once_t0.5', 'facit_pc'),
]

FREQUENCY_METHOD_SPECS = [
    ('detPPR-correctW-every200-after-t0.5', 'det_ppr', 'every200_after_t0.5', 'facit_pc'),
]

DET_PPR_METHOD_SPECS = BASELINE_METHOD_SPECS + SINGLE_HIT_METHOD_SPECS + FREQUENCY_METHOD_SPECS


def benchmark_cases() -> list[GraphCase]:
    selected = {int(g) for g in GRAPH_IDS}
    return [case for case in ACTIVE_CASES if int(case.graph_id) in selected]


def _run_sampler_full_state(*, sampler_kind: str, seed: int, n_steps: int):
    set_seed(int(seed))
    full_state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1.0e-3)),
    )
    last_times = None
    for times in iter_sampling_times(batch=full_state['batch'], grid=full_state['sampling_time_grid']):
        full_state = _benchmark_sampler_step_full_state(
            full_state,
            times,
            sampler_kind=str(sampler_kind),
            tau=float(FACIT_COMPARE_TAU),
            n_correction_steps=int(FACIT_COMPARE_CORRECTION_STEPS),
        )
        last_times = times
    if last_times is None:
        raise RuntimeError('No sampling steps were executed in backend consistency helper.')
    return full_state, last_times


def sampler_backend_consistency_rows(*, seed: int = SAMPLE_SEED, n_steps: int = DET_PPR_OUTER_STEPS) -> list[dict[str, Any]]:
    rows = []
    selected_cases = benchmark_cases()
    if not selected_cases:
        return rows

    def _build_row(case: GraphCase, method_name: str, sampler_kind: str, direct_state: KLDMState, recon_state: KLDMState) -> dict[str, Any]:
        direct_eval = evaluate_arrays(case, direct_state.f, direct_state.l, direct_state.h)
        recon_eval = evaluate_arrays(case, recon_state.f, recon_state.l, recon_state.h)
        f_max_abs = float(torch.max(torch.abs(wrap_residual(direct_state.f, recon_state.f))).detach().item()) if direct_state.f.numel() else 0.0
        v_max_abs = float(torch.max(torch.abs(direct_state.v - recon_state.v)).detach().item()) if direct_state.v.numel() else 0.0
        l_max_abs = float(torch.max(torch.abs(ensure_l_feature(direct_state.l, int(case.gt_frac.shape[0])) - ensure_l_feature(recon_state.l, int(case.gt_frac.shape[0])))).detach().item())
        rmse_direct = direct_eval.get('rmse')
        rmse_recon = recon_eval.get('rmse')
        frac_direct = direct_eval.get('frac_rmse')
        frac_recon = recon_eval.get('frac_rmse')
        rmse_equal = ((rmse_direct is None and rmse_recon is None) or ((rmse_direct is None or not math.isfinite(float(rmse_direct))) and (rmse_recon is None or not math.isfinite(float(rmse_recon)))) or (math.isfinite(float(rmse_direct)) and math.isfinite(float(rmse_recon)) and abs(float(rmse_direct) - float(rmse_recon)) <= 1.0e-8))
        frac_equal = ((frac_direct is None and frac_recon is None) or ((frac_direct is None or not math.isfinite(float(frac_direct))) and (frac_recon is None or not math.isfinite(float(frac_recon)))) or (math.isfinite(float(frac_direct)) and math.isfinite(float(frac_recon)) and abs(float(frac_direct) - float(frac_recon)) <= 1.0e-8))
        pass_flag = bool(
            bool(direct_eval.get('match')) == bool(recon_eval.get('match'))
            and bool(direct_eval.get('valid')) == bool(recon_eval.get('valid'))
            and rmse_equal
            and frac_equal
            and f_max_abs <= 1.0e-8
            and v_max_abs <= 1.0e-8
            and l_max_abs <= 1.0e-8
        )
        return {
            'test': 'sampler_backend_consistency',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'method': method_name,
            'sampler_kind': sampler_kind,
            'direct_match': bool(direct_eval.get('match')),
            'reconstructed_match': bool(recon_eval.get('match')),
            'direct_valid': bool(direct_eval.get('valid')),
            'reconstructed_valid': bool(recon_eval.get('valid')),
            'direct_RMSE': rmse_direct,
            'reconstructed_RMSE': rmse_recon,
            'direct_frac_RMSE': frac_direct,
            'reconstructed_frac_RMSE': frac_recon,
            'f_max_abs': f_max_abs,
            'v_max_abs': v_max_abs,
            'l_max_abs': l_max_abs,
            'PASS': pass_flag,
            'status': 'PASS' if pass_flag else 'FAIL',
        }

    try:
        f_pc, v_pc, l_pc, a_pc = sample_facit_pc_full_batch(seed=int(seed), n_steps=int(n_steps), tau=float(FACIT_COMPARE_TAU), n_correction_steps=int(FACIT_COMPARE_CORRECTION_STEPS))
        recon_pc_full, recon_pc_last_times = _run_sampler_full_state(sampler_kind='facit_pc', seed=int(seed), n_steps=int(n_steps))
        for case in selected_cases:
            g_pc = graph_tensors(case.graph_idx0, source=(f_pc, v_pc, l_pc, a_pc))
            direct_state = KLDMState(f=g_pc['pos'], v=v_pc[graph_slice(case.graph_idx0)[0]:graph_slice(case.graph_idx0)[1]].detach().clone(), l=g_pc['l'], k=l_to_k(g_pc['l'], case), h=g_pc['h'], t=float('nan'), dt=1.0/max(int(n_steps),1), graph_idx0=case.graph_idx0)
            recon_state = final_graph_state_from_full(case, recon_pc_full, recon_pc_last_times)
            rows.append(_build_row(case, 'facitKLDM-PC-1000', 'facit_pc', direct_state, recon_state))
    except Exception as exc:
        rows.append(error_row('sampler_backend_consistency', 'all', exc, 'SAMPLER_BACKEND_PC_ERROR', method='facitKLDM-PC-1000', sampler_kind='facit_pc'))

    try:
        set_seed(int(seed))
        with torch.no_grad():
            f_em, v_em, l_em, a_em = runner.model.sample_CSP_algorithm3_facit(
                n_steps=int(n_steps),
                batch=batch,
                t_start=float(facit_cfg.get('t_start', 1.0)),
                t_final=float(facit_cfg.get('t_final', 1.0e-3)),
            )
        recon_em_full, recon_em_last_times = _run_sampler_full_state(sampler_kind='facit_em', seed=int(seed), n_steps=int(n_steps))
        for case in selected_cases:
            g_em = graph_tensors(case.graph_idx0, source=(f_em, v_em, l_em, a_em))
            direct_state = KLDMState(f=g_em['pos'], v=v_em[graph_slice(case.graph_idx0)[0]:graph_slice(case.graph_idx0)[1]].detach().clone(), l=g_em['l'], k=l_to_k(g_em['l'], case), h=g_em['h'], t=float('nan'), dt=1.0/max(int(n_steps),1), graph_idx0=case.graph_idx0)
            recon_state = final_graph_state_from_full(case, recon_em_full, recon_em_last_times)
            rows.append(_build_row(case, 'facitKLDM-EM-1000', 'facit_em', direct_state, recon_state))
    except Exception as exc:
        rows.append(error_row('sampler_backend_consistency', 'all', exc, 'SAMPLER_BACKEND_EM_ERROR', method='facitKLDM-EM-1000', sampler_kind='facit_em'))

    return rows


def wrapped_frac_rmse(pred_f: torch.Tensor, target_f: torch.Tensor) -> float:
    resid = wrap_residual(pred_f, target_f)
    return float(torch.sqrt(torch.mean(resid ** 2)).detach().item())


def _reverse_exp_step_pf_ode(*, sampling_tdm, f_t: torch.Tensor, v_t: torch.Tensor, score_v: torch.Tensor, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    f_t = sampling_tdm.wrap_displacements(f_t)
    dt_internal = torch.as_tensor(sampling_tdm.T * dt, device=v_t.device, dtype=v_t.dtype)
    exp_dt = torch.exp(dt_internal)
    expm1_dt = torch.expm1(dt_internal)
    score_scale = torch.as_tensor(sampling_tdm.vel_scale ** 2, device=v_t.device, dtype=v_t.dtype)
    v_prev = exp_dt * v_t + score_scale * expm1_dt * score_v
    f_prev = sampling_tdm.wrap_displacements(f_t - dt_internal * v_prev)
    return f_prev, v_prev


@torch.no_grad()
def pf_ode_clean_estimate(case: GraphCase, state: KLDMState, *, n_steps: int = CLEAN_ESTIMATOR_STEPS) -> KLDMCleanEstimate:
    batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
    score_network = runner.model.score_network
    restore_training = score_network.training
    score_network.eval()
    try:
        grid = sampling_grid(
            batch=batch_view,
            n_steps=max(int(n_steps), 1),
            t_start=float(state.t),
            t_final=float(CLEAN_ESTIMATOR_T_FINAL),
        )
        f_t = state.f.detach().clone()
        v_t = state.v.detach().clone()
        l_t = ensure_l_feature(state.l, int(state.f.shape[0])).reshape(1, -1).detach().clone()
        h_t = state.h.detach().clone()
        actual_steps = 0
        for times in iter_sampling_times(batch=batch_view, grid=grid):
            preds = runner.model.score_network(
                t=times.now.graph,
                pos=f_t,
                v=v_t,
                h=h_t,
                l=l_t,
                node_index=batch_view.batch,
                edge_node_index=batch_view.edge_node_index,
            )
            score_v = runner.model.tdm.reconstruct_full_reverse_velocity_score(
                t=times.now.nodes,
                v_t=v_t,
                pred_v=preds['v'],
                index=batch_view.batch,
            )
            f_t, v_t = _reverse_exp_step_pf_ode(
                sampling_tdm=runner.model.tdm,
                f_t=f_t,
                v_t=v_t,
                score_v=score_v,
                dt=times.dt,
            )
            if hasattr(runner.model.diffusion_l, 'reverse_step_predictor'):
                l_t = runner.model.diffusion_l.reverse_step_predictor(
                    t=times.now.lattice,
                    x_t=l_t,
                    pred=preds['l'],
                    dt=times.dt,
                    num_atoms=batch_view.num_atoms,
                )
            actual_steps += 1
        return KLDMCleanEstimate(
            f0_hat=f_t.detach().clone(),
            v0_hat=v_t.detach().clone(),
            l0_hat=l_t.reshape(-1).detach().clone(),
            steps=int(actual_steps),
            estimator_mode='pf_ode_reverse_coords__predictor_lattice',
        )
    finally:
        if restore_training:
            score_network.train()


def deterministic_clean_estimate(
    case: GraphCase,
    state: KLDMState,
    *,
    n_steps: int = CLEAN_ESTIMATOR_STEPS,
    estimator_mode: str = 'deterministic_reverse_predictor',
) -> KLDMCleanEstimate:
    estimator_mode = str(estimator_mode)
    if estimator_mode == 'pf_ode':
        return pf_ode_clean_estimate(case, state, n_steps=int(n_steps))
    if estimator_mode == 'facit_pc_mixed_debug':
        batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
        return debug_mixed_clean_estimate(
            model=runner.model,
            batch=batch_view,
            state=to_algo16_state(state),
            node_index=batch_view.batch,
            edge_node_index=batch_view.edge_node_index,
            n_steps=int(n_steps),
            t_start=float(state.t),
            t_final=float(CLEAN_ESTIMATOR_T_FINAL),
        )
    batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
    return deterministic_predictor_clean_estimate(
        model=runner.model,
        batch=batch_view,
        state=to_algo16_state(state),
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        n_steps=int(n_steps),
        t_start=float(state.t),
        t_final=float(CLEAN_ESTIMATOR_T_FINAL),
    )


def sample_forward_noisy_gt_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED) -> KLDMState:
    ref_state = sample_kldm_state_for_graph_at_t_threshold(
        case,
        t_threshold=float(t_value),
        n_steps=int(DET_PPR_OUTER_STEPS),
        seed=int(seed),
        sampler_kind='facit_pc',
    )
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed))
    renoise = kldm_forward_renoise_fv_only(
        model=runner.model,
        batch=batch_view,
        f0=case.gt_frac,
        l0=ensure_l_feature(case.gt_lattice, int(case.gt_frac.shape[0])),
        t_graph=torch.as_tensor([float(ref_state.t)], device=case.gt_frac.device, dtype=case.gt_frac.dtype),
        node_index=batch_view.batch,
        v0=None,
        mean_free_velocity=True,
    )
    return KLDMState(
        f=renoise.f_t.detach().clone(),
        v=renoise.v_t.detach().clone(),
        l=ensure_l_feature(case.gt_lattice, int(case.gt_frac.shape[0])).detach().clone(),
        k=l_to_k(ensure_l_feature(case.gt_lattice, int(case.gt_frac.shape[0])), case),
        h=case.atomic_numbers.detach().clone(),
        t=float(ref_state.t),
        dt=float(ref_state.dt),
        graph_idx0=case.graph_idx0,
        full_state=None,
        full_times=None,
    )


def projection_safety(proj) -> tuple[bool, list[str]]:
    reasons: list[str] = []
    if not bool(torch.isfinite(proj.z_f).all()):
        reasons.append('projected_f_nonfinite')
    if not bool(proj.projection_success):
        reasons.append('projection_failed')
    if float(proj.idempotence_error) > 1.0e-5:
        reasons.append('projection_not_idempotent')
    if bool(proj.assignment_changed):
        reasons.append('assignment_changed')
    if bool(proj.branch_changed):
        reasons.append('branch_changed')
    if not math.isfinite(float(proj.objective)):
        reasons.append('projection_objective_nonfinite')
    return (len(reasons) == 0), reasons


def deterministic_denoiser_sanity_rows(case: GraphCase, *, t_values: tuple[float, ...] | None = None, seed: int = SAMPLE_SEED) -> list[dict[str, Any]]:
    rows = []
    for t_value in (DET_COMPARE_T_VALUES if t_values is None else t_values):
        for estimator_mode, estimator_label in DETERMINISTIC_ESTIMATOR_SPECS:
            try:
                state = sample_forward_noisy_gt_state(case, t_value=float(t_value), seed=int(seed))
                clean = deterministic_clean_estimate(case, state, estimator_mode=str(estimator_mode))
                eval_clean = evaluate_arrays(case, clean.f0_hat, case.gt_lattice, case.atomic_numbers)
                rows.append({
                    'test': 'deterministic_denoiser_sanity',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't_value': float(t_value),
                    'denoiser_mode': str(estimator_mode),
                    'denoiser_label': str(estimator_label),
                    'estimator_mode': str(estimator_mode),
                    'frac_rmse_to_gt': float(wrapped_frac_rmse(clean.f0_hat, case.gt_frac)),
                    'match': bool(eval_clean['match']),
                    'valid': bool(eval_clean['valid']),
                    'RMSE': eval_clean['rmse'],
                    'frac_RMSE': eval_clean['frac_rmse'],
                    'min_pair_distance': eval_clean['min_pair_distance'],
                    'PASS': bool(eval_clean['valid'] and eval_clean['match']),
                    'status': 'PASS' if bool(eval_clean['valid'] and eval_clean['match']) else 'WARN',
                })
            except Exception as exc:
                rows.append(error_row('deterministic_denoiser_sanity', case.graph_id, exc, 'DETERMINISTIC_DENOISER_SANITY_ERROR', space_group=case.requested_sg, t_value=float(t_value), denoiser_mode=str(estimator_mode), estimator_mode=str(estimator_mode)))
    return rows


def select_best_deterministic_estimator(df: pd.DataFrame) -> dict[str, Any]:
    if df.empty or 'denoiser_mode' not in df.columns:
        return {'mode': 'deterministic_reverse_predictor', 'reason': 'No deterministic estimator rows available.'}
    valid = df.copy()
    valid = valid[valid['denoiser_mode'].notna()]
    if valid.empty:
        return {'mode': 'deterministic_reverse_predictor', 'reason': 'No deterministic estimator rows available.'}
    grp = valid.groupby('denoiser_mode', as_index=False).agg({
        'match': 'mean',
        'valid': 'mean',
        'RMSE': 'mean',
        'frac_RMSE': 'mean',
        'frac_rmse_to_gt': 'mean',
    })
    grp = grp.rename(columns={'match': 'match_rate', 'valid': 'valid_rate', 'RMSE': 'mean_RMSE', 'frac_RMSE': 'mean_frac_RMSE', 'frac_rmse_to_gt': 'mean_frac_rmse_to_gt'})
    grp = grp.sort_values(['match_rate', 'valid_rate', 'mean_RMSE', 'mean_frac_rmse_to_gt'], ascending=[False, False, True, True]).reset_index(drop=True)
    best = grp.iloc[0]
    return {
        'mode': str(best['denoiser_mode']),
        'summary': grp,
        'reason': f"Selected {best['denoiser_mode']} by deterministic phase ranking: highest match/valid, then lowest RMSE.",
    }


def _template_candidate_fixtures_for_space_group(case: GraphCase, *, space_group_number: int, cache_key_prefix: str = 'template_candidates_sg') -> list[FixedTemplateFixture]:
    key = (cache_key_prefix, int(case.graph_id), int(space_group_number))
    if key in _caches:
        return _caches[key]
    num_atoms = int(case.gt_frac.shape[0])
    target_cell = cell_from_l(case.gt_lattice, num_atoms).to(device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    oracle_structure = _build_vanilla_structure(
        frac_coords=case.gt_frac,
        atomic_numbers=case.atomic_numbers,
        cell_matrix=target_cell,
    )

    def _select_candidates(standardization: str):
        return select_requested_template_states(
            frac_coords=case.gt_frac,
            atomic_numbers=case.atomic_numbers,
            cell_matrix=target_cell,
            space_group_number=int(space_group_number),
            standardization=standardization,
            symprec=1e-2,
            angle_tolerance=5.0,
            max_templates=max(FIXTURE_MAX_TEMPLATES, 16),
            template_eval_limit=max(FIXTURE_TEMPLATE_EVAL_LIMIT, 6),
            optimization_steps=FIXTURE_OPT_STEPS,
            learning_rate=FIXTURE_LR,
            coord_weight=1.0,
            lattice_weight=0.0,
            pairdist_weight=0.0,
            steric_weight=0.0,
            volume_weight=0.0,
            k6_weight=0.0,
            freeze_lattice_free_vars=True,
            quick_templates=False,
            top_k=max(TEMPLATE_TOP_K, 6),
            template_prior=template_prior,
            template_prior_weight=0.0,
            oracle_reference_structure=oracle_structure,
            oracle_fit_structure=oracle_structure,
        )

    candidate_entries = []
    for standardization in ['conventional', 'primitive']:
        try:
            states = _select_candidates(standardization)
        except Exception:
            continue
        for state0 in states:
            chart_frac, chart_atomic_numbers, chart_cell, chart_k, target_repr = _pcs_state_chart_target(state0)
            if int(chart_atomic_numbers.shape[0]) != num_atoms:
                continue
            candidate_entries.append({
                'standardization': standardization,
                'state': state0,
                'chart_frac': chart_frac,
                'chart_atomic_numbers': chart_atomic_numbers,
                'chart_cell': chart_cell,
                'chart_k': chart_k,
                'target_repr': target_repr,
            })

    fixtures_out: list[tuple[tuple[Any, ...], FixedTemplateFixture]] = []
    for item in candidate_entries:
        state0 = item['state']
        tau0 = torch.zeros(1, 3, device=item['chart_frac'].device, dtype=item['chart_frac'].dtype)
        fixed_assignment = state0.fixed_target_assignment if state0.fixed_target_assignment is not None else state0.anchor_assignment
        if fixed_assignment is None:
            continue
        try:
            projection = project_to_fixed_template_local(
                f_frac=item['chart_frac'],
                atomic_numbers=item['chart_atomic_numbers'],
                template_state=state0,
                target_k=item['chart_k'],
                tau0=tau0,
                theta0=state0.free_vars,
                fixed_assignment=fixed_assignment,
                config=FIXED_CFG,
            )
            validation = validate_projection(case, projection)
            coord_residual = float(torch.linalg.norm(wrap_residual(item['chart_frac'], projection.z_frac).reshape(-1)).detach().item())
            score = (
                0 if validation.requested_space_group_match else 1,
                0 if validation.composition_match else 1,
                0 if item['standardization'] == 'conventional' else 1,
                coord_residual,
                float(projection.objective),
            )
            fixture = FixedTemplateFixture(
                case=case,
                state=projection.raw.state,
                theta=projection.theta.detach().clone(),
                tau=projection.tau.detach().clone(),
                assignment=projection.assignment.detach().clone(),
                target_k=item['chart_k'].detach().clone(),
                z_frac=projection.z_frac.detach().clone(),
                z_l=projection.z_l.detach().clone(),
                chart_atomic_numbers=item['chart_atomic_numbers'].detach().clone(),
                template_labels=template_labels(projection.raw.state.template),
                template_multiplicities=template_multiplicities(projection.raw.state.template),
                template_dofs=template_dofs(projection.raw.state.template),
                template_total_dof=int(projection.raw.state.template.total_free_dims),
                requested_sg_match=bool(validation.requested_space_group_match),
                composition_match=bool(validation.composition_match),
                projection_objective=float(projection.objective),
                candidate_standardization=item['standardization'],
            )
            fixtures_out.append((score, fixture))
        except Exception:
            continue

    fixtures_out.sort(key=lambda pair: pair[0])
    out = [fixture for _, fixture in fixtures_out]
    _caches[key] = out
    return out


def _fixed_template_candidate_fixtures(case: GraphCase) -> list[FixedTemplateFixture]:
    return _template_candidate_fixtures_for_space_group(case, space_group_number=int(case.requested_sg), cache_key_prefix='fixed_template_candidates')


def wrong_space_group_fixture(case: GraphCase) -> FixedTemplateFixture | None:
    candidate_sgs = [1, 2, 3, 4, 5, 123, 221]
    for sg in candidate_sgs:
        if int(sg) == int(case.requested_sg):
            continue
        candidates = _template_candidate_fixtures_for_space_group(case, space_group_number=int(sg), cache_key_prefix='wrong_space_group_candidates')
        if candidates:
            return candidates[0]
    return None


def random_same_multiplicity_fixture(case: GraphCase) -> FixedTemplateFixture | None:
    base = fixtures[case.graph_id]
    candidates = [fixture for fixture in _fixed_template_candidate_fixtures(case) if fixture.template_multiplicities == base.template_multiplicities and fixture.template_labels != base.template_labels]
    if not candidates:
        return None
    return candidates[-1]


def random_projected_template_projection(case: GraphCase, clean: KLDMCleanEstimate) -> Any:
    fixture = fixtures[case.graph_id]
    gen = torch.Generator(device=clean.f0_hat.device)
    gen.manual_seed(int(SAMPLE_SEED) + int(case.graph_id) + 12345)
    random_frac = torch.rand(clean.f0_hat.shape, generator=gen, device=clean.f0_hat.device, dtype=clean.f0_hat.dtype)
    return project_clean_to_fixed_wyckoff(
        f0_hat=random_frac,
        atomic_numbers=case.atomic_numbers,
        template_state=fixture.state,
        target_k=fixture.target_k,
        tau0=fixture.tau,
        theta0=fixture.theta,
        fixed_assignment=fixture.assignment,
        config=FIXED_CFG,
        reference_frac=random_frac,
    )


def alternate_same_sg_fixture(case: GraphCase) -> FixedTemplateFixture | None:
    candidates = _fixed_template_candidate_fixtures(case)
    if len(candidates) < 2:
        return None
    return candidates[1]


def shuffled_assignment_like(assignment: torch.Tensor | None) -> torch.Tensor | None:
    if assignment is None or int(assignment.numel()) <= 1:
        return None
    return assignment.reshape(-1).roll(1).reshape_as(assignment)


def shuffled_species_like(h: torch.Tensor) -> torch.Tensor | None:
    h = h.reshape(-1).to(torch.long)
    if int(h.numel()) <= 1 or int(torch.unique(h).numel()) <= 1:
        return None
    candidate = h.flip(0)
    if torch.equal(candidate, h):
        candidate = h.roll(1)
    return candidate.detach().clone()


def project_clean_with_control(
    case: GraphCase,
    graph_state: KLDMState,
    clean: KLDMCleanEstimate,
    *,
    variant: str,
) -> dict[str, Any]:
    fixture = fixtures[case.graph_id]
    atomic_numbers = graph_state.h
    assignment = fixture.assignment
    control_reason = ''
    if variant == 'det_ppr_wrong_W_same_space_group':
        alt = alternate_same_sg_fixture(case)
        if alt is None:
            return {'available': False, 'reason': 'wrong_same_sg_fixture_unavailable', 'projection': None, 'fixture': None, 'atomic_numbers': None}
        fixture = alt
        assignment = fixture.assignment
        control_reason = 'alternate_same_space_group_template'
    elif variant == 'det_ppr_wrong_space_group_W':
        alt = wrong_space_group_fixture(case)
        if alt is None:
            return {'available': False, 'reason': 'wrong_space_group_fixture_unavailable', 'projection': None, 'fixture': None, 'atomic_numbers': None}
        fixture = alt
        assignment = fixture.assignment
        control_reason = 'wrong_space_group_template'
    elif variant == 'det_ppr_random_W_same_multiplicity':
        alt = random_same_multiplicity_fixture(case)
        if alt is None:
            return {'available': False, 'reason': 'random_same_multiplicity_fixture_unavailable', 'projection': None, 'fixture': None, 'atomic_numbers': None}
        fixture = alt
        assignment = fixture.assignment
        control_reason = 'random_same_multiplicity_template'
    elif variant == 'det_ppr_wrong_assignment':
        shuffled = shuffled_assignment_like(fixture.assignment)
        if shuffled is None:
            return {'available': False, 'reason': 'wrong_assignment_unavailable', 'projection': None, 'fixture': None, 'atomic_numbers': None}
        assignment = shuffled
        control_reason = 'rolled_fixed_assignment'
    elif variant == 'det_ppr_species_shuffled_W':
        shuffled_species = shuffled_species_like(graph_state.h)
        if shuffled_species is None:
            return {'available': False, 'reason': 'species_shuffle_unavailable', 'projection': None, 'fixture': None, 'atomic_numbers': None}
        atomic_numbers = shuffled_species
        control_reason = 'species_shuffled_projection_input'
    elif variant == 'det_ppr_random_projected_template':
        proj = random_projected_template_projection(case, clean)
        return {
            'available': True,
            'reason': 'random_projected_template',
            'projection': proj,
            'fixture': fixture,
            'atomic_numbers': atomic_numbers,
        }
    proj = project_clean_to_fixed_wyckoff(
        f0_hat=clean.f0_hat,
        atomic_numbers=atomic_numbers,
        template_state=fixture.state,
        target_k=fixture.target_k,
        tau0=fixture.tau,
        theta0=fixture.theta,
        fixed_assignment=assignment,
        config=FIXED_CFG,
        reference_frac=clean.f0_hat,
    )
    return {
        'available': True,
        'reason': control_reason,
        'projection': proj,
        'fixture': fixture,
        'atomic_numbers': atomic_numbers,
    }


def projection_clean_residual_rows(case: GraphCase, *, t_values: tuple[float, ...] | None = None, seed: int = SAMPLE_SEED, estimator_mode: str = 'deterministic_reverse_predictor') -> list[dict[str, Any]]:
    rows = []
    for t_value in (DET_PPR_ONCE_T_VALUES if t_values is None else t_values):
        try:
            state = sample_kldm_state_for_graph_at_t_threshold(case, t_threshold=float(t_value), n_steps=int(DET_PPR_OUTER_STEPS), seed=int(seed), sampler_kind='facit_pc')
            clean = deterministic_clean_estimate(case, state, estimator_mode=str(estimator_mode))
            proj = gate_projection(case, state, clean)
            eval_proj = evaluate_arrays(case, proj.z_f, state.l, case.atomic_numbers)
            safe, reasons = projection_safety(proj)
            rows.append({
                'test': 'projection_improves_clean_residual',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't_value': float(t_value),
                'F0_hat_finite': bool(torch.isfinite(clean.f0_hat).all()),
                'F0_C_finite': bool(torch.isfinite(proj.z_f).all()),
                'D_proj': float(projection_distance_to_fixture(proj.z_f, clean.f0_hat)),
                'projected_finite': bool(torch.isfinite(proj.z_f).all()),
                'projection_success': bool(proj.projection_success),
                'idempotence_error': float(proj.idempotence_error),
                'assignment_changed': bool(proj.assignment_changed),
                'branch_changed': bool(proj.branch_changed),
                'projected_min_pair_distance': eval_proj['min_pair_distance'],
                'reject_reasons': ','.join(reasons),
                'PASS': bool(safe),
                'status': 'PASS' if bool(safe) else 'WARN',
            })
        except Exception as exc:
            rows.append(error_row('projection_improves_clean_residual', case.graph_id, exc, 'PROJECTION_CLEAN_RESIDUAL_ERROR', space_group=case.requested_sg, t_value=float(t_value)))
    return rows


def final_generated_pairwise_frac_rmse(method_a: str, method_b: str, graph_id: int) -> float:
    artifact_a = result_artifacts.get((str(method_a), int(graph_id)))
    artifact_b = result_artifacts.get((str(method_b), int(graph_id)))
    if artifact_a is None or artifact_b is None:
        return float('nan')
    frac_a = artifact_a['final_frac']
    frac_b = artifact_b['final_frac']
    return float(torch.sqrt(torch.mean(wrap_residual(frac_a, frac_b) ** 2)).detach().item())


def final_template_residual_rows(case: GraphCase, *, methods: list[str] | None = None, t_value: float = 0.5) -> list[dict[str, Any]]:
    rows = []
    method_names = methods if methods is not None else [
        'facitKLDM-EM-1000',
        'facitKLDM-PC-1000',
        'detPPR-correctW-once-t0.5',
        'detPPR-wrongSG-once-t0.5',
        'detPPR-correctW-every200-after-t0.5',
    ]
    variant_pairs = {
        'det_ppr_wrong_space_group_W': 'wrongSG',
    }
    for method_name in method_names:
        artifact = result_artifacts.get((str(method_name), int(case.graph_id)))
        if artifact is None:
            continue
        final_frac = artifact['final_frac']
        final_h = artifact['final_h']
        correct_proj = project_clean_to_fixed_wyckoff(
            f0_hat=final_frac,
            atomic_numbers=final_h,
            template_state=fixtures[case.graph_id].state,
            target_k=fixtures[case.graph_id].target_k,
            tau0=fixtures[case.graph_id].tau,
            theta0=fixtures[case.graph_id].theta,
            fixed_assignment=fixtures[case.graph_id].assignment,
            config=FIXED_CFG,
            reference_frac=final_frac,
        )
        d_correct = float(torch.linalg.norm(wrap_residual(final_frac, correct_proj.z_f).reshape(-1)).detach().item())
        for control_variant, control_label in variant_pairs.items():
            try:
                control = project_clean_with_control(case, KLDMState(f=final_frac, v=torch.zeros_like(final_frac), l=artifact['final_l'], k=l_to_k(artifact['final_l'], case), h=final_h, t=float(t_value), dt=1.0/max(int(DET_PPR_OUTER_STEPS),1), graph_idx0=case.graph_idx0), KLDMCleanEstimate(f0_hat=final_frac, v0_hat=torch.zeros_like(final_frac), l0_hat=artifact['final_l'], steps=0, estimator_mode='final_sample_probe'), variant=control_variant)
                if not bool(control.get('available', False)):
                    rows.append({'test':'final_template_residuals','graph':case.graph_id,'space_group':case.requested_sg,'method':method_name,'control_variant':control_label,'D_correct':d_correct,'D_wrong':float('nan'),'D_correct_lt_D_wrong':False,'PASS':False,'status':'WARN','control_reason':control.get('reason','')})
                    continue
                wrong_proj = control['projection']
                d_wrong = float(torch.linalg.norm(wrap_residual(final_frac, wrong_proj.z_f).reshape(-1)).detach().item())
                rows.append({'test':'final_template_residuals','graph':case.graph_id,'space_group':case.requested_sg,'method':method_name,'control_variant':control_label,'D_correct':d_correct,'D_wrong':d_wrong,'D_correct_lt_D_wrong':bool(d_correct < d_wrong),'PASS':bool(d_correct < d_wrong),'status':'PASS' if bool(d_correct < d_wrong) else 'WARN','control_reason':control.get('reason','')})
            except Exception as exc:
                rows.append(error_row('final_template_residuals', case.graph_id, exc, 'FINAL_TEMPLATE_RESIDUALS_ERROR', method=method_name, control_variant=control_label, space_group=case.requested_sg))
    return rows


def anchor_control_comparison_rows(case: GraphCase, *, estimator_mode: str, t_value: float = 0.5) -> list[dict[str, Any]]:
    rows = []
    state = sample_kldm_state_for_graph_at_t_threshold(case, t_threshold=float(t_value), n_steps=int(DET_PPR_OUTER_STEPS), seed=int(SAMPLE_SEED), sampler_kind='facit_pc')
    clean = deterministic_clean_estimate(case, state, estimator_mode=str(estimator_mode))
    correct = gate_projection(case, state, clean)
    method_pairs = {
        'det_ppr_wrong_W_same_space_group': ('detPPR-correctW-once-t0.5', 'detPPR-wrongW-sameSG-once-t0.5'),
        'det_ppr_wrong_assignment': ('detPPR-correctW-once-t0.5', 'detPPR-wrongAssignment-once-t0.5'),
        'det_ppr_species_shuffled_W': ('detPPR-correctW-once-t0.5', 'detPPR-speciesShuffledW-once-t0.5'),
        'det_ppr_wrong_space_group_W': ('detPPR-correctW-once-t0.5', 'detPPR-wrongSG-once-t0.5'),
        'det_ppr_random_W_same_multiplicity': ('detPPR-correctW-once-t0.5', None),
        'det_ppr_random_projected_template': ('detPPR-correctW-once-t0.5', None),
    }
    for variant, (method_correct, method_wrong) in method_pairs.items():
        try:
            control = project_clean_with_control(case, state, clean, variant=variant)
            if not bool(control.get('available', False)):
                rows.append({
                    'test': 'anchor_control_comparison',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't_value': float(t_value),
                    'control_variant': variant,
                    'anchor_distance': float('nan'),
                    'final_generated_frac_rmsd': float('nan'),
                    'control_available': False,
                    'control_reason': control.get('reason', ''),
                    'PASS': False,
                    'status': 'WARN',
                })
                continue
            wrong_proj = control['projection']
            anchor_distance = float(torch.linalg.norm(wrap_residual(correct.z_f, wrong_proj.z_f).reshape(-1)).detach().item())
            final_rmsd = float('nan') if method_wrong is None else final_generated_pairwise_frac_rmse(method_correct, method_wrong, case.graph_id)
            rows.append({
                'test': 'anchor_control_comparison',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't_value': float(t_value),
                'control_variant': variant,
                'anchor_distance': anchor_distance,
                'final_generated_frac_rmsd': final_rmsd,
                'control_available': True,
                'control_reason': control.get('reason', ''),
                'PASS': bool(math.isfinite(anchor_distance) and anchor_distance > 1.0e-8),
                'status': 'PASS' if bool(math.isfinite(anchor_distance) and anchor_distance > 1.0e-8) else 'WARN',
            })
        except Exception as exc:
            rows.append(error_row('anchor_control_comparison', case.graph_id, exc, 'ANCHOR_CONTROL_COMPARISON_ERROR', space_group=case.requested_sg, t_value=float(t_value), control_variant=variant))
    return rows


def _evaluate_clean_after(case: GraphCase, clean_after: KLDMCleanEstimate, lattice_features: torch.Tensor) -> dict[str, Any]:
    try:
        eval_clean_after = evaluate_arrays(case, clean_after.f0_hat, lattice_features, case.atomic_numbers)
        rmse_val = eval_clean_after['rmse']
        rmse_failed = rmse_val is None or not math.isfinite(float(rmse_val))
        return {
            'valid_after_denoise': bool(eval_clean_after['valid']),
            'RMSE_after_denoise': float(rmse_val) if (rmse_val is not None and math.isfinite(float(rmse_val))) else float('nan'),
            'RMSE_after_denoise_eval_failed': bool(rmse_failed),
            'after_eval_error_type': None,
            'after_eval_error_message': None,
        }
    except Exception as exc:
        return {
            'valid_after_denoise': False,
            'RMSE_after_denoise': float('nan'),
            'RMSE_after_denoise_eval_failed': True,
            'after_eval_error_type': type(exc).__name__,
            'after_eval_error_message': str(exc),
        }



def _safe_float(value: Any, default: float = float('nan')) -> float:
    if value is None:
        return default
    try:
        out = float(value)
    except (TypeError, ValueError):
        return default
    return out


def det_ppr_no_harm_gate(attempt: dict[str, Any]) -> tuple[bool, list[str]]:
    reasons: list[str] = []
    if not bool(attempt.get('projected_finite', False)):
        reasons.append('projected_clean_nonfinite')
    if not bool(attempt.get('projection_safe', False)):
        reasons.append('projection_unsafe')
    if not bool(attempt.get('renoise_finite_ok', False)):
        reasons.append('renoise_nonfinite')
    mean_free_norm = _safe_float(attempt.get('mean_free_norm'), float('inf'))
    if not (math.isfinite(mean_free_norm) and mean_free_norm < float(DET_PPR_MEAN_FREE_THRESHOLD)):
        reasons.append('velocity_not_mean_free')
    if not bool(attempt.get('f_v_kernel_consistency_ok', False)):
        reasons.append('f_v_kernel_inconsistent')
    lattice_changed_norm = _safe_float(attempt.get('lattice_changed_norm_from_state'), float('inf'))
    if not (math.isfinite(lattice_changed_norm) and lattice_changed_norm <= 1.0e-8):
        reasons.append('lattice_changed')
    min_dist = _safe_float(attempt.get('active_projected_min_pair_distance'), float('nan'))
    if not (math.isfinite(min_dist) and min_dist > float(MIN_PAIR_DISTANCE_GUARD)):
        reasons.append('projected_clean_min_pair_distance_unsafe')
    d_after = _safe_float(attempt.get('D_after'), float('inf'))
    d_before = _safe_float(attempt.get('D_before'), float('-inf'))
    if not (math.isfinite(d_after) and math.isfinite(d_before) and d_after < d_before):
        reasons.append('D_after_not_improved')
    return len(reasons) == 0, reasons


def correct_wrongsg_anchor_rows(case: GraphCase, *, estimator_mode: str, t_value: float = 0.5) -> list[dict[str, Any]]:
    rows = []
    state = sample_kldm_state_for_graph_at_t_threshold(case, t_threshold=float(t_value), n_steps=int(DET_PPR_OUTER_STEPS), seed=int(SAMPLE_SEED), sampler_kind='facit_pc')
    clean = deterministic_clean_estimate(case, state, estimator_mode=str(estimator_mode))
    correct_proj = gate_projection(case, state, clean)
    wrong = project_clean_with_control(case, state, clean, variant='det_ppr_wrong_space_group_W')
    if not bool(wrong.get('available', False)):
        rows.append({
            'test': 'correct_vs_wrongSG_anchor_comparison',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            't_value': float(t_value),
            'PASS': False,
            'status': 'WARN',
            'control_available': False,
            'control_reason': wrong.get('reason', ''),
        })
        return rows
    wrong_proj = wrong['projection']
    eval_correct = evaluate_arrays(case, correct_proj.z_f, state.l, case.atomic_numbers)
    eval_wrong = evaluate_arrays(case, wrong_proj.z_f, state.l, case.atomic_numbers)
    gen = torch.Generator(device=state.f.device)
    gen.manual_seed(int(SAMPLE_SEED) + 1000 + int(case.graph_id))
    noise_v = torch.randn(tuple(state.f.shape), generator=gen, device=state.f.device, dtype=state.f.dtype)
    noise_r = torch.randn(tuple(state.f.shape), generator=gen, device=state.f.device, dtype=state.f.dtype)
    batch_view = make_single_graph_batch_view(case, ref_tensor=state.f)
    t_graph = torch.as_tensor([float(state.t)], device=state.f.device, dtype=state.f.dtype)
    renoise_correct = kldm_forward_renoise_fv_only(model=runner.model, batch=batch_view, f0=correct_proj.z_f, l0=state.l, t_graph=t_graph, node_index=batch_view.batch, v0=None, noise_v=noise_v, noise_r=noise_r, mean_free_velocity=True)
    renoise_wrong = kldm_forward_renoise_fv_only(model=runner.model, batch=batch_view, f0=wrong_proj.z_f, l0=state.l, t_graph=t_graph, node_index=batch_view.batch, v0=None, noise_v=noise_v, noise_r=noise_r, mean_free_velocity=True)
    rows.append({
        'test': 'correct_vs_wrongSG_anchor_comparison',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        't_value': float(t_value),
        'anchor_RMSE_correctW': eval_correct['rmse'],
        'anchor_RMSE_wrongSG': eval_wrong['rmse'],
        'anchor_frac_RMSE_correctW': eval_correct['frac_rmse'],
        'anchor_frac_RMSE_wrongSG': eval_wrong['frac_rmse'],
        'anchor_distance_correct_vs_wrong': float(torch.linalg.norm(wrap_residual(correct_proj.z_f, wrong_proj.z_f).reshape(-1)).detach().item()),
        'post_renoise_distance_correct_vs_wrong': float(torch.linalg.norm(wrap_residual(renoise_correct.f_t, renoise_wrong.f_t).reshape(-1)).detach().item()),
        'final_structure_RMSD_correct_vs_wrong': final_generated_pairwise_frac_rmse('detPPR-correctW-once-t0.5', 'detPPR-wrongSG-once-t0.5', case.graph_id),
        'correct_anchor_better_than_wrongSG': bool((eval_correct['rmse'] is not None and eval_wrong['rmse'] is not None and math.isfinite(float(eval_correct['rmse'])) and math.isfinite(float(eval_wrong['rmse'])) and float(eval_correct['rmse']) < float(eval_wrong['rmse']))) if (eval_correct['rmse'] is not None and eval_wrong['rmse'] is not None) else False,
        'control_reason': wrong.get('reason', ''),
        'PASS': True,
        'status': 'PASS',
    })
    return rows


def apply_det_ppr(
    case: GraphCase,
    graph_state: KLDMState,
    *,
    clean_steps: int = CLEAN_ESTIMATOR_STEPS,
    variant: str = 'det_ppr',
    estimator_mode: str = 'deterministic_reverse_predictor',
) -> dict[str, Any]:
    batch_view = make_single_graph_batch_view(case, ref_tensor=graph_state.f)
    clean = deterministic_clean_estimate(case, graph_state, n_steps=int(clean_steps), estimator_mode=str(estimator_mode))
    attempt = {
        'variant': variant,
        'clean_nonfinite': not bool(torch.isfinite(clean.f0_hat).all()),
        'projection_applied': False,
        'projection_safe': False,
        'control_available': True,
        'control_reason': '',
        'renoise_finite_ok': False,
        'mean_free_norm': float('nan'),
        'D_before': float('nan'),
        'D_after': float('nan'),
        'D_delta': float('nan'),
        'D_after_to_initial_anchor': float('nan'),
        'lattice_changed_norm': float('nan'),
        'epsilon_v_mean_norm': float('nan'),
        'v_t_mean_norm_before_optional_centering': float('nan'),
        'v_t_mean_norm_after': float('nan'),
        'f_v_kernel_consistency_ok': False,
        'projection_shock': float('nan'),
        'active_projected_min_pair_distance': float('nan'),
        'RMSE_after_denoise': float('nan'),
        'RMSE_after_denoise_eval_failed': False,
        'valid_after_denoise': False,
    }
    if attempt['clean_nonfinite']:
        return {'accepted': False, 'reject_reason': 'clean_nonfinite', 'attempt_metrics': attempt, 'accepted_result': None}

    proj_oracle = gate_projection(case, graph_state, clean)
    attempt['D_before'] = float(projection_distance_to_fixture(proj_oracle.z_f, clean.f0_hat))
    attempt['oracle_projection_safe'], oracle_reasons = projection_safety(proj_oracle)
    attempt['projection_idempotence_error'] = float(proj_oracle.idempotence_error)
    attempt['assignment_changed'] = bool(proj_oracle.assignment_changed)
    attempt['branch_changed'] = bool(proj_oracle.branch_changed)
    attempt['projected_finite'] = bool(torch.isfinite(proj_oracle.z_f).all())
    try:
        attempt['projected_min_pair_distance'] = evaluate_arrays(case, proj_oracle.z_f, graph_state.l, case.atomic_numbers)['min_pair_distance']
    except Exception:
        attempt['projected_min_pair_distance'] = float('nan')

    target_f0 = clean.f0_hat.detach().clone()
    active_projection = proj_oracle
    if variant == 'det_denoise_renoise_no_projection':
        attempt['control_reason'] = 'no_projection_control'
    else:
        control = project_clean_with_control(case, graph_state, clean, variant=variant)
        if not bool(control['available']):
            attempt['control_available'] = False
            attempt['control_reason'] = str(control['reason'])
            return {'accepted': False, 'reject_reason': str(control['reason']), 'attempt_metrics': attempt, 'accepted_result': None}
        active_projection = control['projection']
        target_f0 = active_projection.z_f.detach().clone()
        attempt['projection_applied'] = True
        attempt['projection_shock'] = float(projection_distance_to_fixture(active_projection.z_f, clean.f0_hat))
        attempt['control_reason'] = str(control['reason'])
        safe, reasons = projection_safety(active_projection)
        attempt['projection_safe'] = bool(safe)
        attempt['projection_reject_reasons'] = ';'.join(reasons)
        attempt['projected_finite'] = bool(torch.isfinite(active_projection.z_f).all())
        try:
            attempt['active_projected_min_pair_distance'] = evaluate_arrays(case, active_projection.z_f, graph_state.l, case.atomic_numbers)['min_pair_distance']
        except Exception:
            attempt['active_projected_min_pair_distance'] = float('nan')
        if not safe:
            return {'accepted': False, 'reject_reason': ';'.join(reasons), 'attempt_metrics': attempt, 'accepted_result': None}

    renoise = kldm_forward_renoise_fv_only(
        model=runner.model,
        batch=batch_view,
        f0=target_f0,
        l0=graph_state.l,
        t_graph=torch.as_tensor([float(graph_state.t)], device=graph_state.f.device, dtype=graph_state.f.dtype),
        node_index=batch_view.batch,
        v0=None,
        mean_free_velocity=True,
    )
    attempt['renoise_finite_ok'] = bool(renoise.finite_ok)
    attempt['mean_free_norm'] = float(renoise.mean_free_norm)
    attempt['epsilon_v_mean_norm'] = float(getattr(renoise, 'epsilon_v_mean_norm', float('nan')))
    attempt['v_t_mean_norm_before_optional_centering'] = float(getattr(renoise, 'v_t_mean_norm_before_optional_centering', float('nan')))
    attempt['v_t_mean_norm_after'] = float(getattr(renoise, 'v_t_mean_norm_after', float('nan')))
    attempt['f_v_kernel_consistency_ok'] = bool(getattr(renoise, 'f_v_kernel_consistency_ok', False))
    attempt['lattice_changed_norm'] = float(getattr(renoise, 'lattice_changed_norm', float('nan')))
    attempt['lattice_changed_norm_from_state'] = float(torch.linalg.norm((renoise.l_t.reshape(-1) - graph_state.l.reshape(-1))).detach().item())
    if not bool(renoise.finite_ok):
        return {'accepted': False, 'reject_reason': 'renoise_nonfinite', 'attempt_metrics': attempt, 'accepted_result': None}
    if not (float(renoise.mean_free_norm) < float(DET_PPR_MEAN_FREE_THRESHOLD)):
        return {'accepted': False, 'reject_reason': 'mean_free_norm_too_large', 'attempt_metrics': attempt, 'accepted_result': None}

    state_after = KLDMState(
        f=renoise.f_t.detach().clone(),
        v=renoise.v_t.detach().clone(),
        l=graph_state.l.detach().clone(),
        k=graph_state.k.detach().clone(),
        h=graph_state.h.detach().clone(),
        t=float(graph_state.t),
        dt=float(graph_state.dt),
        graph_idx0=graph_state.graph_idx0,
        full_state=None,
        full_times=None,
    )
    clean_after = deterministic_clean_estimate(case, state_after, n_steps=int(clean_steps), estimator_mode=str(estimator_mode))
    proj_after = gate_projection(case, state_after, clean_after)
    attempt['D_after'] = float(projection_distance_to_fixture(proj_after.z_f, clean_after.f0_hat))
    attempt['D_after_to_initial_anchor'] = float(projection_distance_to_fixture(proj_oracle.z_f, clean_after.f0_hat))
    attempt['D_delta'] = float(attempt['D_after'] - attempt['D_before'])
    attempt.update(_evaluate_clean_after(case, clean_after, graph_state.l))
    no_harm_ok, no_harm_reasons = det_ppr_no_harm_gate(attempt)
    attempt['no_harm_gate_pass'] = bool(no_harm_ok)
    attempt['no_harm_gate_reasons'] = ';'.join(no_harm_reasons)
    if not no_harm_ok:
        return {'accepted': False, 'reject_reason': ';'.join(no_harm_reasons), 'attempt_metrics': attempt, 'accepted_result': None}

    return {
        'accepted': True,
        'reject_reason': '',
        'attempt_metrics': attempt,
        'accepted_result': {
            'state_out': state_after,
            'clean_before': clean,
            'projection_before': active_projection,
            'clean_after': clean_after,
            'projection_after': proj_after,
            'renoise': renoise,
        },
    }


def one_step_project_renoise_sanity_rows(case: GraphCase, *, t_values: tuple[float, ...] | None = None, seed: int = SAMPLE_SEED) -> list[dict[str, Any]]:
    rows = []
    for t_value in (DET_PPR_ONCE_T_VALUES if t_values is None else t_values):
        try:
            state = sample_kldm_state_for_graph_at_t_threshold(case, t_threshold=float(t_value), n_steps=int(DET_PPR_OUTER_STEPS), seed=int(seed), sampler_kind='facit_pc')
            outcome = apply_det_ppr(case, state, clean_steps=int(CLEAN_ESTIMATOR_STEPS), variant='det_ppr', estimator_mode=str(BEST_DETERMINISTIC_ESTIMATOR_MODE))
            metrics = outcome['attempt_metrics']
            pass_flag = bool(
                outcome['accepted']
                and metrics.get('renoise_finite_ok', False)
                and _safe_float(metrics.get('mean_free_norm'), float('inf')) < float(DET_PPR_MEAN_FREE_THRESHOLD)
                and bool(metrics.get('valid_after_denoise', False))
                and (not bool(metrics.get('RMSE_after_denoise_eval_failed', False)))
                and _safe_float(metrics.get('D_after'), float('inf')) < _safe_float(metrics.get('D_before'), float('-inf'))
            )
            status = 'PASS' if pass_flag else 'WARN'
            rows.append({
                'test': 'one_step_project_renoise_sanity',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't_value': float(t_value),
                'D_before': metrics.get('D_before'),
                'D_after': metrics.get('D_after'),
                'D_delta': metrics.get('D_delta'),
                'D_after_to_initial_anchor': metrics.get('D_after_to_initial_anchor'),
                'renoise_finite_ok': bool(metrics.get('renoise_finite_ok', False)),
                'mean_free_norm': metrics.get('mean_free_norm'),
                'epsilon_v_mean_norm': metrics.get('epsilon_v_mean_norm'),
                'v_t_mean_norm_before_optional_centering': metrics.get('v_t_mean_norm_before_optional_centering'),
                'v_t_mean_norm_after': metrics.get('v_t_mean_norm_after'),
                'f_v_kernel_consistency_ok': metrics.get('f_v_kernel_consistency_ok'),
                'lattice_changed_norm': metrics.get('lattice_changed_norm'),
                'lattice_changed_norm_from_state': metrics.get('lattice_changed_norm_from_state'),
                'valid_after_denoise': metrics.get('valid_after_denoise'),
                'RMSE_after_denoise': metrics.get('RMSE_after_denoise'),
                'RMSE_after_denoise_eval_failed': metrics.get('RMSE_after_denoise_eval_failed'),
                'accepted': bool(outcome['accepted']),
                'reject_reason': outcome['reject_reason'],
                'PASS': pass_flag,
                'status': status,
            })
        except Exception as exc:
            rows.append(error_row('one_step_project_renoise_sanity', case.graph_id, exc, 'ONE_STEP_PROJECT_RENOISE_SANITY_ERROR', space_group=case.requested_sg, t_value=float(t_value)))
    return rows


def _parse_det_trigger_mode(trigger_mode: str) -> tuple[str, float | None]:
    mode = str(trigger_mode).strip().lower()
    if mode.startswith('once_t'):
        return 'once', float(mode.split('once_t', 1)[1])
    if mode.startswith('every200_after_t'):
        return 'every200_after', float(mode.split('every200_after_t', 1)[1])
    if mode.startswith('every100_after_t'):
        return 'every100_after', float(mode.split('every100_after_t', 1)[1])
    raise ValueError(f'Unsupported deterministic trigger_mode={trigger_mode!r}')


def should_trigger_det_ppr(*, step_idx: int, trigger_mode: str, t_graph: float, already_triggered: bool) -> bool:
    kind, threshold = _parse_det_trigger_mode(trigger_mode)
    if threshold is None or not math.isfinite(float(t_graph)):
        return False
    if float(t_graph) > float(threshold):
        return False
    if kind == 'once':
        return not bool(already_triggered)
    if kind in {'every100_after', 'every200_after'}:
        return int(step_idx) % int(DET_PPR_EVERY_K_STEPS) == 0
    return False


def _summarize_det_attempts(logs: list[dict[str, Any]]) -> dict[str, Any]:
    if not logs:
        return {
            'attempt_step_indices': '',
            'attempt_D_delta_mean': float('nan'),
            'attempt_mean_free_norm_max': float('nan'),
            'attempt_lattice_changed_norm_max': float('nan'),
            'attempt_epsilon_v_mean_norm_max': float('nan'),
            'attempt_v_t_mean_norm_after_max': float('nan'),
            'attempt_projection_shock_mean': float('nan'),
            'accepted_step_indices': '',
            'accepted_hits': 0,
        }
    d_deltas = [float(log['metrics'].get('D_delta')) for log in logs if math.isfinite(float(log['metrics'].get('D_delta', float('nan'))))]
    mean_free = [float(log['metrics'].get('mean_free_norm')) for log in logs if math.isfinite(float(log['metrics'].get('mean_free_norm', float('nan'))))]
    lattice_changed = [float(log['metrics'].get('lattice_changed_norm')) for log in logs if math.isfinite(float(log['metrics'].get('lattice_changed_norm', float('nan'))))]
    eps_mean = [float(log['metrics'].get('epsilon_v_mean_norm')) for log in logs if math.isfinite(float(log['metrics'].get('epsilon_v_mean_norm', float('nan'))))]
    vt_mean_after = [float(log['metrics'].get('v_t_mean_norm_after')) for log in logs if math.isfinite(float(log['metrics'].get('v_t_mean_norm_after', float('nan'))))]
    proj_shock = [float(log['metrics'].get('projection_shock')) for log in logs if math.isfinite(float(log['metrics'].get('projection_shock', float('nan'))))]
    return {
        'attempt_step_indices': ','.join(str(int(log['step_idx'])) for log in logs),
        'attempt_D_delta_mean': float(np.mean(d_deltas)) if d_deltas else float('nan'),
        'attempt_mean_free_norm_max': float(np.max(mean_free)) if mean_free else float('nan'),
        'attempt_lattice_changed_norm_max': float(np.max(lattice_changed)) if lattice_changed else float('nan'),
        'attempt_epsilon_v_mean_norm_max': float(np.max(eps_mean)) if eps_mean else float('nan'),
        'attempt_v_t_mean_norm_after_max': float(np.max(vt_mean_after)) if vt_mean_after else float('nan'),
        'attempt_projection_shock_mean': float(np.mean(proj_shock)) if proj_shock else float('nan'),
        'accepted_step_indices': ','.join(str(int(log['step_idx'])) for log in logs if bool(log['accepted'])),
        'accepted_hits': int(sum(1 for log in logs if bool(log['accepted']))),
    }


def future_xt_projection_through_denoiser_placeholder(*args, **kwargs):
    raise NotImplementedError('Future xt-space projection-through-denoiser PPR is intentionally not active in this notebook.')


def final_graph_state_from_full(case: GraphCase, full_state: dict[str, Any], last_times) -> KLDMState:
    if hasattr(last_times, 'next'):
        final_times = SimpleNamespace(now=last_times.next, dt=last_times.dt)
    else:
        final_times = last_times
    return graph_state_from_full(clone_full_state(full_state), case, times=final_times)


def run_det_ppr_benchmark_variant(
    method_name: str,
    *,
    variant: str,
    trigger_mode: str | None,
    sampler_kind: str = 'facit_pc',
    graph_ids: list[int] | None = None,
    seed: int = SAMPLE_SEED,
    n_steps: int = DET_PPR_OUTER_STEPS,
    clean_steps: int = CLEAN_ESTIMATOR_STEPS,
    estimator_mode: str = 'deterministic_reverse_predictor',
) -> list[dict[str, Any]]:
    set_seed(int(seed))
    selected_graph_ids = {int(g) for g in (graph_ids if graph_ids is not None else [case.graph_id for case in benchmark_cases()])}
    selected_cases = [case for case in benchmark_cases() if int(case.graph_id) in selected_graph_ids]
    if not selected_cases:
        raise RuntimeError(f'No ACTIVE_CASES matched graph_ids={sorted(selected_graph_ids)}')

    started_at = time.perf_counter()
    full_state = runner.model._prepare_csp_sampling(
        batch=batch,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get('t_start', 1.0)),
        t_final=float(facit_cfg.get('t_final', 1.0e-3)),
    )
    activation_hits = {case.graph_id: 0 for case in selected_cases}
    projection_success_hits = {case.graph_id: 0 for case in selected_cases}
    renoise_hits = {case.graph_id: 0 for case in selected_cases}
    accepted_hits = {case.graph_id: 0 for case in selected_cases}
    reject_reasons = {case.graph_id: [] for case in selected_cases}
    attempt_logs = {case.graph_id: [] for case in selected_cases}
    once_triggered = {case.graph_id: False for case in selected_cases}
    last_times = None

    for step_idx, times in enumerate(iter_sampling_times(batch=full_state['batch'], grid=full_state['sampling_time_grid']), start=1):
        full_state = _benchmark_sampler_step_full_state(
            full_state,
            times,
            sampler_kind=str(sampler_kind),
            tau=float(FACIT_COMPARE_TAU),
            n_correction_steps=int(FACIT_COMPARE_CORRECTION_STEPS),
        )
        last_times = times
        if variant in {'baseline', 'final_projection_only'}:
            continue
        current_times = SimpleNamespace(now=times.next, dt=times.dt)
        for case in selected_cases:
            graph_state = graph_state_from_full(clone_full_state(full_state), case, times=current_times)
            if trigger_mode is None:
                continue
            if not should_trigger_det_ppr(step_idx=int(step_idx), trigger_mode=str(trigger_mode), t_graph=float(graph_state.t), already_triggered=bool(once_triggered[case.graph_id])):
                continue
            activation_hits[case.graph_id] += 1
            kind, _threshold = _parse_det_trigger_mode(str(trigger_mode))
            if kind == 'once':
                once_triggered[case.graph_id] = True
            outcome = apply_det_ppr(case, graph_state, clean_steps=int(clean_steps), variant=str(variant), estimator_mode=str(estimator_mode))
            attempt_logs[case.graph_id].append({
                'step_idx': int(step_idx),
                'accepted': bool(outcome['accepted']),
                'reject_reason': outcome['reject_reason'],
                'metrics': outcome['attempt_metrics'],
            })
            if bool(outcome['attempt_metrics'].get('projection_applied', False)) and bool(outcome['attempt_metrics'].get('projection_safe', False)):
                projection_success_hits[case.graph_id] += 1
            if not outcome['accepted']:
                reject_reasons[case.graph_id].append(outcome['reject_reason'])
                continue
            renoise_hits[case.graph_id] += 1
            accepted_hits[case.graph_id] += 1
            accepted = outcome['accepted_result']
            write_graph_state_into_full_state(full_state, case, f=accepted['state_out'].f, v=accepted['state_out'].v, l=graph_state.l, h=graph_state.h)

    runtime = float(time.perf_counter() - started_at)
    if last_times is None:
        raise RuntimeError('No sampling steps were executed.')

    rows = []
    for case in selected_cases:
        final_state = final_graph_state_from_full(case, full_state, last_times)
        final_projection_result = None
        if variant == 'final_projection_only':
            fixture = fixtures[case.graph_id]
            final_projection_result = guarded_final_local_projection(
                case,
                final_state,
                template_state=fixture.state,
                target_k=fixture.target_k,
                tau0=fixture.tau,
                theta0=fixture.theta,
                fixed_assignment=fixture.assignment,
            )
            eval_final = final_projection_result['eval_proj'] if bool(final_projection_result['accepted']) else final_projection_result['eval_unproj']
        else:
            eval_final = evaluate_arrays(case, final_state.f, final_state.l, final_state.h)
        attempt_summary = _summarize_det_attempts(attempt_logs[case.graph_id])
        ppr_applied = bool(accepted_hits[case.graph_id] > 0 or (final_projection_result is not None and bool(final_projection_result['accepted'])))
        final_projection_applied = bool(final_projection_result is not None)
        final_projection_accepted = bool(final_projection_result['accepted']) if final_projection_result is not None else False
        projection_shock = float(final_projection_result['shock']) if final_projection_result is not None else float('nan')
        row_pass = True
        row_status = 'PASS'
        if variant not in {'baseline', 'final_projection_only'}:
            if int(activation_hits[case.graph_id]) == 0:
                row_pass = False
                row_status = 'FAIL'
            elif int(accepted_hits[case.graph_id]) == 0:
                row_pass = False
                row_status = 'WARN'
        result_artifacts[(str(method_name), int(case.graph_id))] = {
            'final_frac': final_state.f.detach().clone(),
            'final_l': final_state.l.detach().clone(),
            'final_h': final_state.h.detach().clone(),
            'variant': variant,
            'sampler_kind': sampler_kind,
        }
        rows.append({
            'method': method_name,
            'sample_id': case.graph_id,
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'match': bool(eval_final['match']),
            'RMSE': eval_final['rmse'],
            'frac_RMSE': eval_final['frac_rmse'],
            'valid_structure': bool(eval_final['valid']),
            'close_contact': bool((eval_final['min_pair_distance'] is not None) and math.isfinite(float(eval_final['min_pair_distance'])) and float(eval_final['min_pair_distance']) <= float(MIN_PAIR_DISTANCE_GUARD)),
            'min_pair_distance': eval_final['min_pair_distance'],
            'volume_ratio': eval_final['volume_ratio'],
            'requested_sg_match_diagnostic': eval_final['requested_sg_match'],
            'runtime': runtime / max(len(selected_cases), 1),
            'activation_hits': int(activation_hits[case.graph_id]),
            'projection_success_hits': int(projection_success_hits[case.graph_id]),
            'renoise_hits': int(renoise_hits[case.graph_id]),
            'renoise_success_hits': int(renoise_hits[case.graph_id]),
            'accepted_hits': int(accepted_hits[case.graph_id]),
            'reject_reasons': ','.join(reject_reasons[case.graph_id]) if reject_reasons[case.graph_id] else '',
            'variant': variant,
            'trigger_mode': trigger_mode,
            'sampler_kind': sampler_kind,
            't_hit': _parse_det_trigger_mode(str(trigger_mode))[1] if trigger_mode is not None else None,
            'clean_estimator_steps': int(clean_steps),
            'implemented_mode': variant,
            'PPR_applied': ppr_applied,
            'projection_shock': projection_shock,
            'final_projection_applied': final_projection_applied,
            'final_projection_accepted': final_projection_accepted,
            'final_projection_reject_reason': None if final_projection_result is None else final_projection_result['rejected_reason'],
            **attempt_summary,
            'PASS': row_pass,
            'status': row_status,
        })
    return rows


def ppr_attempt_display_columns() -> list[str]:
    return [
        'method',
        'graph',
        'sampler_kind',
        'activation_hits',
        'projection_success_hits',
        'renoise_success_hits',
        'accepted_hits',
        'reject_reasons',
        'attempt_step_indices',
        'attempt_D_delta_mean',
        'attempt_mean_free_norm_max',
        'attempt_lattice_changed_norm_max',
        'attempt_epsilon_v_mean_norm_max',
        'attempt_v_t_mean_norm_after_max',
        'attempt_projection_shock_mean',
        'accepted_step_indices',
        'final_projection_accepted',
        'PPR_applied',
        'RMSE',
        'frac_RMSE',
        'match',
        'valid_structure',
    ]


def single_hit_frequency_gate(df_single_hit: pd.DataFrame) -> tuple[bool, str]:
    if df_single_hit.empty or 'method' not in df_single_hit.columns:
        return False, 'single-hit table is empty'
    sample_rows = df_single_hit[df_single_hit.get('sample_id', pd.Series(dtype=object)).notna()].copy() if 'sample_id' in df_single_hit.columns else pd.DataFrame()
    if sample_rows.empty:
        return False, 'single-hit table has no sample rows'
    baseline = sample_rows[sample_rows['method'] == PRIMARY_BASELINE_METHOD]
    ppr_rows = sample_rows[sample_rows['method'] == 'detPPR-once-t0.5']
    if baseline.empty or ppr_rows.empty:
        return False, 'PC baseline or detPPR-once-t0.5 rows are missing'
    baseline_by_graph = {int(row['graph']): row for _, row in baseline.iterrows()}
    for _, row in ppr_rows.iterrows():
        graph = int(row['graph'])
        base = baseline_by_graph.get(graph)
        if base is None:
            continue
        if not bool(row.get('PPR_applied', False)):
            continue
        if not bool(row.get('valid_structure', False)) or bool(row.get('close_contact', True)):
            continue
        match_ok = bool(row.get('match', False)) or not bool(base.get('match', False))
        rmse_ok = math.isfinite(float(row.get('RMSE', float('nan')))) and math.isfinite(float(base.get('RMSE', float('nan')))) and float(row['RMSE']) <= float(base['RMSE']) + 1.0e-12
        if match_ok and rmse_ok:
            return True, f"single-hit {row['method']} worked on graph {graph}"
    return False, 'detPPR-once-t0.5 did not preserve/improve the PC baseline'


def final_recommendation_from_aggregate(df_benchmark: pd.DataFrame) -> dict[str, Any]:
    baseline = None
    if not df_benchmark.empty and (df_benchmark['method'] == PRIMARY_BASELINE_METHOD).any():
        baseline = df_benchmark[df_benchmark['method'] == PRIMARY_BASELINE_METHOD].iloc[0]
    elif not df_benchmark.empty:
        baseline_rows = df_benchmark[df_benchmark['method'].astype(str).str.startswith('facitKLDM-')]
        baseline = baseline_rows.iloc[0] if not baseline_rows.empty else None
    if baseline is None:
        return {'baseline': None, 'best_match': None, 'best_rmse': None, 'use_det_ppr': False, 'reason': 'No baseline row available.'}
    ppr_only = df_benchmark[~df_benchmark['method'].astype(str).str.startswith('facitKLDM-')].copy() if not df_benchmark.empty else pd.DataFrame()
    best_match = ppr_only.sort_values(['mean_match_rate', 'mean_RMSE'], ascending=[False, True]).iloc[0] if not ppr_only.empty else None
    eligible = ppr_only[np.abs(ppr_only['mean_match_rate'] - float(baseline['mean_match_rate'])) <= 1.0e-12] if not ppr_only.empty else pd.DataFrame()
    better_match = ppr_only[ppr_only['mean_match_rate'] > float(baseline['mean_match_rate']) + 1.0e-12] if not ppr_only.empty else pd.DataFrame()
    rmse_pool = pd.concat([better_match, eligible], axis=0).drop_duplicates() if not ppr_only.empty else pd.DataFrame()
    best_rmse = rmse_pool.sort_values(['mean_match_rate', 'mean_RMSE'], ascending=[False, True]).iloc[0] if not rmse_pool.empty else None
    use_det_ppr = bool(best_match is not None and bool(best_match['beats_baseline_overall']))
    reason = f"Deterministic KLDM-Wyckoff clean-project-renoise improves CSP-level performance over {PRIMARY_BASELINE_METHOD}." if use_det_ppr else f"Deterministic KLDM-Wyckoff clean-project-renoise does not improve CSP-level match_rate/RMSE over {PRIMARY_BASELINE_METHOD}. Keep the baseline sampler."
    return {
        'baseline': baseline,
        'best_match': best_match,
        'best_rmse': best_rmse,
        'use_det_ppr': use_det_ppr,
        'reason': reason,
    }


## Phase 1 — Baselines (EM / PC)


In [ ]:
if RUN_SAMPLER_BACKEND_CONSISTENCY:
    rows_backend = sampler_backend_consistency_rows(seed=SAMPLE_SEED, n_steps=int(DET_PPR_OUTER_STEPS))
    df_sampler_backend = dataframe_result('test_0_sampler_backend_consistency', rows_backend)
else:
    df_sampler_backend = dataframe_result('test_0_sampler_backend_consistency', [{'test': 'sampler_backend_consistency', 'graph': 'all', 'PASS': False, 'status': 'SKIPPED', 'reason': 'RUN_SAMPLER_BACKEND_CONSISTENCY=0'}])
display(df_sampler_backend)


In [ ]:

rows = []
for method_name, variant, trigger_mode, sampler_kind in BASELINE_METHOD_SPECS:
    try:
        rows.extend(run_det_ppr_benchmark_variant(
            method_name,
            variant=variant,
            trigger_mode=trigger_mode,
            sampler_kind=sampler_kind,
            graph_ids=[case.graph_id for case in benchmark_cases()],
            seed=SAMPLE_SEED,
            n_steps=int(DET_PPR_OUTER_STEPS),
            clean_steps=int(CLEAN_ESTIMATOR_STEPS),
        ))
    except Exception as exc:
        rows.append(error_row('test_A_baseline_sanity', method_name, exc, 'BASELINE_SANITY_ERROR', method=method_name, sampler_kind=sampler_kind))

df_test_A_baseline = dataframe_result('test_A_baseline_sanity', rows)
display(df_test_A_baseline)

df_test_A_baseline_agg = aggregate_benchmark(df_test_A_baseline[df_test_A_baseline['sample_id'].notna()].copy()) if ('sample_id' in df_test_A_baseline.columns and df_test_A_baseline['sample_id'].notna().any()) else pd.DataFrame()
display(df_test_A_baseline_agg)


## Step 12 — Optional Final Projection Note


In [ ]:

optional_final_projection_policy = pd.DataFrame([{
    'step': 'Step 12 optional final projection',
    'raw_sampler_output_is_primary_metric': True,
    'final_projection_in_active_sampler': False,
    'reason': 'The benchmark judges raw generated samples unless a final projection is explicitly made part of the method.',
}])
result_tables['optional_final_projection_policy'] = optional_final_projection_policy
display(optional_final_projection_policy)


## Phase 2 — Deterministic Sampler Comparison


In [ ]:

rows_denoiser = []
rows_projection = []
rows_one_step = []
for case in benchmark_cases():
    rows_denoiser.extend(deterministic_denoiser_sanity_rows(case))

df_test_D_deterministic_denoiser = dataframe_result('test_D_deterministic_denoiser_quality', rows_denoiser)
display(df_test_D_deterministic_denoiser)

det_choice = select_best_deterministic_estimator(df_test_D_deterministic_denoiser)
result_tables['deterministic_estimator_choice'] = det_choice['summary'] if 'summary' in det_choice else pd.DataFrame()
BEST_DETERMINISTIC_ESTIMATOR_MODE = 'pf_ode'
print(f"BEST_DETERMINISTIC_ESTIMATOR_MODE={BEST_DETERMINISTIC_ESTIMATOR_MODE}")
print('Phase 3 is forced to use PF-ODE by experiment design; Phase 2 still reports the comparison table.')
print(det_choice['reason'])
if 'summary' in det_choice:
    display(det_choice['summary'])

for case in benchmark_cases():
    rows_projection.extend(projection_clean_residual_rows(case, estimator_mode=str(BEST_DETERMINISTIC_ESTIMATOR_MODE)))
    rows_one_step.extend(one_step_project_renoise_sanity_rows(case))

df_test_E_projection_clean = dataframe_result('test_E_clean_projection_sanity', rows_projection)
display(df_test_E_projection_clean)

df_test_F_one_step = dataframe_result('test_F_one_step_project_renoise', rows_one_step)
display(df_test_F_one_step)


## Phase 3 — PPR Benchmarks Using Best Deterministic Estimator


In [ ]:

rows_single_hit = []
for method_name, variant, trigger_mode, sampler_kind in SINGLE_HIT_METHOD_SPECS:
    try:
        rows_single_hit.extend(run_det_ppr_benchmark_variant(
            method_name,
            variant=variant,
            trigger_mode=trigger_mode,
            sampler_kind=sampler_kind,
            graph_ids=[case.graph_id for case in benchmark_cases()],
            seed=SAMPLE_SEED,
            n_steps=int(DET_PPR_OUTER_STEPS),
            clean_steps=int(CLEAN_ESTIMATOR_STEPS),
            estimator_mode=str(BEST_DETERMINISTIC_ESTIMATOR_MODE),
        ))
    except Exception as exc:
        rows_single_hit.append(error_row('test_G_single_hit_full_chain', method_name, exc, 'SINGLE_HIT_FULL_CHAIN_ERROR', method=method_name, trigger_mode=trigger_mode, sampler_kind=sampler_kind))

df_test_G_single_hit_methods = dataframe_result('test_G_single_hit_methods_only', rows_single_hit)
df_test_G_single_hit = dataframe_result('test_G_single_hit_full_chain', list(df_test_A_baseline.to_dict('records')) + rows_single_hit)
display(df_test_G_single_hit)
cols = [c for c in ppr_attempt_display_columns() if c in df_test_G_single_hit.columns]
display(df_test_G_single_hit[cols])

rows_correct_wrongsg_anchor = []
for case in benchmark_cases():
    rows_correct_wrongsg_anchor.extend(correct_wrongsg_anchor_rows(case, estimator_mode=str(BEST_DETERMINISTIC_ESTIMATOR_MODE), t_value=0.5))
df_test_G_correct_wrongSG_anchor = dataframe_result('test_G_correct_wrongSG_anchor_comparison', rows_correct_wrongsg_anchor)
display(df_test_G_correct_wrongSG_anchor)

rows_final_template_residuals = []
for case in benchmark_cases():
    rows_final_template_residuals.extend(final_template_residual_rows(case, t_value=0.5))
df_test_G_final_template_residuals = dataframe_result('test_G_final_template_residuals', rows_final_template_residuals)
display(df_test_G_final_template_residuals)

test_G_sample_rows = df_test_G_single_hit[df_test_G_single_hit['sample_id'].notna()].copy() if ('sample_id' in df_test_G_single_hit.columns and df_test_G_single_hit['sample_id'].notna().any()) else pd.DataFrame()
df_test_H_activation_sweep = aggregate_benchmark(test_G_sample_rows) if not test_G_sample_rows.empty else pd.DataFrame()
result_tables['test_H_activation_time_sweep'] = df_test_H_activation_sweep
display(df_test_H_activation_sweep)

run_frequency_sweep, frequency_gate_reason = True, 'forced_on_for_phase_3_experiment'
print(f'frequency_sweep_gate={run_frequency_sweep} reason={frequency_gate_reason}')
rows_frequency = []
for method_name, variant, trigger_mode, sampler_kind in FREQUENCY_METHOD_SPECS:
    try:
        rows_frequency.extend(run_det_ppr_benchmark_variant(
            method_name,
            variant=variant,
            trigger_mode=trigger_mode,
            sampler_kind=sampler_kind,
            graph_ids=[case.graph_id for case in benchmark_cases()],
            seed=SAMPLE_SEED,
            n_steps=int(DET_PPR_OUTER_STEPS),
            clean_steps=int(CLEAN_ESTIMATOR_STEPS),
            estimator_mode=str(BEST_DETERMINISTIC_ESTIMATOR_MODE),
        ))
    except Exception as exc:
        rows_frequency.append(error_row('test_I_frequency_sweep', method_name, exc, 'FREQUENCY_SWEEP_ERROR', method=method_name, trigger_mode=trigger_mode, sampler_kind=sampler_kind))

df_test_I_frequency_methods = dataframe_result('test_I_frequency_methods_only', rows_frequency)
df_test_I_frequency = dataframe_result('test_I_frequency_sweep', list(df_test_A_baseline.to_dict('records')) + rows_frequency)
display(df_test_I_frequency)
cols = [c for c in ppr_attempt_display_columns() if c in df_test_I_frequency.columns]
if cols:
    display(df_test_I_frequency[cols])

test_I_sample_rows = df_test_I_frequency[df_test_I_frequency['sample_id'].notna()].copy() if ('sample_id' in df_test_I_frequency.columns and df_test_I_frequency['sample_id'].notna().any()) else pd.DataFrame()
df_test_I_frequency_agg = aggregate_benchmark(test_I_sample_rows) if not test_I_sample_rows.empty else pd.DataFrame()
result_tables['test_I_frequency_sweep_aggregate'] = df_test_I_frequency_agg
display(df_test_I_frequency_agg)


## Final Benchmark And Recommendation


In [ ]:

benchmark_rows = []
for name in ['test_A_baseline_sanity', 'test_G_single_hit_methods_only', 'test_I_frequency_methods_only']:
    df_src = result_tables.get(name, pd.DataFrame())
    if not df_src.empty:
        benchmark_rows.extend(df_src.to_dict('records'))

df_det_ppr_benchmark_samples = dataframe_result('det_ppr_benchmark_samples', benchmark_rows)
display(df_det_ppr_benchmark_samples)
cols = [c for c in ppr_attempt_display_columns() if c in df_det_ppr_benchmark_samples.columns]
if cols:
    display(df_det_ppr_benchmark_samples[cols])
det_ppr_sample_rows = df_det_ppr_benchmark_samples[df_det_ppr_benchmark_samples['sample_id'].notna()].copy() if (not df_det_ppr_benchmark_samples.empty and 'sample_id' in df_det_ppr_benchmark_samples.columns) else pd.DataFrame()
df_det_ppr_benchmark = aggregate_benchmark(det_ppr_sample_rows) if not det_ppr_sample_rows.empty else pd.DataFrame()
result_tables['det_ppr_benchmark_aggregate'] = df_det_ppr_benchmark
display(df_det_ppr_benchmark)
rec = final_recommendation_from_aggregate(df_det_ppr_benchmark) if not df_det_ppr_benchmark.empty else {'baseline': None, 'best_match': None, 'best_rmse': None, 'use_det_ppr': False, 'reason': 'No benchmark rows available.'}
baseline = rec['baseline']
best_match = rec['best_match']
best_rmse = rec['best_rmse']
print('Method class: x0-space clean-project-renoise approximation')
print('Full PPR = False')
print('Uses denoiser differentiation = False')
print('Uses KLDM-native forward renoise = True')
print('Lattice projected = False')
print('Clean velocity policy: V0_C = 0')
print(f'Best deterministic estimator used in Phase 3 = {BEST_DETERMINISTIC_ESTIMATOR_MODE}')
print('')
print(f'PRIMARY BASELINE = {PRIMARY_BASELINE_METHOD}')
print('')
if baseline is not None:
    print('Primary baseline row:')
    print(f"    method = {baseline['method']}")
    print(f"    mean_match_rate = {float(baseline['mean_match_rate'])}")
    print(f"    mean_RMSE = {float(baseline['mean_RMSE'])}")
    print('')
print('Best deterministic project-renoise method by mean match rate:')
print(f"    method = {None if best_match is None else best_match['method']}")
print(f"    mean_match_rate = {None if best_match is None else float(best_match['mean_match_rate'])}")
print(f"    mean_RMSE = {None if best_match is None else float(best_match['mean_RMSE'])}")
print(f"    beats_baseline = {None if best_match is None else bool(best_match['beats_baseline_overall'])}")
print('')
print('Best deterministic project-renoise method by RMSE among methods with baseline-or-better match rate:')
print(f"    method = {None if best_rmse is None else best_rmse['method']}")
print(f"    mean_match_rate = {None if best_rmse is None else float(best_rmse['mean_match_rate'])}")
print(f"    mean_RMSE = {None if best_rmse is None else float(best_rmse['mean_RMSE'])}")
print(f"    beats_baseline = {None if best_rmse is None else bool(best_rmse['beats_baseline_overall'])}")
print('')
print('Recommendation:')
print(f"    use_detKLDM_Wyckoff_CPR = {bool(rec['use_det_ppr'])}")
print(f"    reason = {rec['reason']}")
